# Defending Image Classification Systems Against Evasion Attacks
### A Comparative Study of Adversarial Training Methods
**Morris Asasira A. | S24B23/116 | Uganda Christian University**

**The idea in one paragraph.** A normal image classifier can be tricked: change an image by a tiny amount that
no human can see, and the model suddenly gives the wrong answer. This notebook trains one residual CNN
architecture with three different defenses (**SAT**, **Ensemble AT** and **TRADES**), attacks every trained model
with the same **FGSM** and **PGD** attacks, and compares the results under identical conditions.

> **No results are stored in this notebook.** All outputs were cleared when the notebook was upgraded to the new
> architecture. Every number, table and chart is produced only when you run the cells yourself.

# 1. Research Overview and Experimental Objectives

**Research objectives (from the proposal, Section 1.3)**

1. **Objective 1:** Implement and train three defenses (Standard Adversarial Training, Ensemble Adversarial Training and TRADES)
   with the *same* CNN architecture on the *same* dataset, so they can be compared directly and fairly.
2. **Objective 2:** Test the robustness of the trained models with FGSM and PGD, reporting **robust (adversarial) accuracy**
   and **attack success rate (ASR)**.
3. **Objective 3:** Study the trade-offs between **clean accuracy**, **robustness** and **training time**.

**Experimental design (Proposal Section 3.1).** This is a controlled comparative experiment.
The **defense method is the only independent variable**. Everything else is held constant:

| Held constant | How this notebook does it |
|---|---|
| Architecture | One `ResCNN` class, created by one `create_model()` function, for every model |
| Data split | One fixed train / validation split (seeded and saved to disk); the official CIFAR-10 test set is used only for final evaluation |
| Optimiser, batch size, epochs, LR schedule | One shared configuration cell and one shared training loop |
| Random seeds | The same list of seeds `SEEDS` for every defense |
| Training attack budget | All defenses use the same $\epsilon$, step size and number of steps |
| Evaluation attacks | Identical FGSM / PGD settings, identical test images and identical random starts for every model |

**Dependent variables (Proposal Section 3.5):** clean accuracy, robust accuracy (always reported with the attack, its source and $\epsilon$),
attack success rate, training time, plus the loss, confidence and calibration measures added in this notebook.

**Change from the proposal.** Proposal Section 3.2 names the shared network `SimpleCNN`. This version of the notebook replaces it with a
moderately deep residual network, `ResCNN` (Section 4), which extracts features better. The experimental logic is unchanged
because *all* defenses still share the one architecture. Section 3.2 of the proposal should be updated to match.

### Key terms used in this notebook

| Term | Meaning |
|---|---|
| **Perturbation** $\delta$ | The small change added to an image: $x_{adv} = x + \delta$ |
| **$L_\infty$ budget $\epsilon$** | No single pixel value may change by more than $\epsilon$. Pixels live in $[0, 1]$, so $\epsilon = 8/255 \approx 0.031$ means "at most 8 brightness levels out of 255" |
| **White-box attack** | The attacker can see the target model's weights and uses *its* gradients |
| **Transfer (black-box) attack** | Adversarial images are made on a *different* model (here: the undefended Standard model) and then shown to the target |
| **Clean accuracy** | Accuracy on the original, unmodified test images |
| **Robust accuracy** | Accuracy on attacked test images |
| **Attack success rate (ASR)** | Of the images the model got right *before* the attack, the percentage it gets wrong *after* the attack |
| **ECE** | Expected Calibration Error: how far the model's confidence is from its real accuracy (Section 15) |

### Notebook map

| Section | What it does | Needs a GPU? |
|---|---|---|
| 1 | Research overview (this section) | No |
| 2 | Imports, Google Drive and the experiment configuration | No |
| 3 | CIFAR-10 loading, fixed data split and preprocessing | No |
| 4 | The shared `ResCNN` architecture | No |
| 5–6 | FGSM and PGD attack implementations (+ a check against `torchattacks`) | No |
| 7–9 | The three defenses: SAT, Ensemble AT, TRADES | No (definitions only) |
| 10 | Training all models (**slow**) | **Yes** |
| 11 | Checkpoint check and loading the trained models | No |
| 12–14 | Clean, FGSM and PGD evaluation (white-box and transfer) | Yes (recommended) |
| 15 | Confidence and calibration analysis | No |
| 16 | Comparison tables, charts, reports, confusion matrices | No |
| 17 | Observations and conclusions (you write these) | No |
| Appendix | Interactive demos from the original notebook | Optional |

**How to use it**

1. In Google Colab choose *Runtime → Change runtime type → GPU*.
2. **First do a quick test run:** set `QUICK_RUN = True` in Section 2.5 and run the whole notebook. It trains tiny versions of every model
   for 1 epoch on 2,000 images, so you can check that everything works end-to-end in a few minutes. Quick-run files go into a separate folder
   and never mix with the real experiment. (The quick-run numbers mean nothing scientifically.)
3. Set `QUICK_RUN = False` and run the notebook from top to bottom for the real experiment. Training saves a checkpoint after every epoch,
   so if Colab disconnects you can just run the notebook again and training continues where it stopped.

---
# 2. Imports and Configuration
Run every cell in this section first, in order.

**2.1 Check the hardware.**
This cell asks the computer whether it has a GPU (a fast chip used for training).
Training adversarially robust models without a GPU is not practical.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

**2.2 Connect to Google Drive.**
Checkpoints and results are saved to the `adv_demo` folder on Drive so they survive when Colab disconnects.
If you run the notebook outside Colab (for example in local Jupyter), a local folder called `adv_demo_local` is used instead.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/adv_demo"
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = os.path.abspath("adv_demo_local")
    IN_COLAB = False

os.makedirs(PROJECT_DIR, exist_ok=True)
print("Running in Colab:", IN_COLAB)
print("Project folder:  ", PROJECT_DIR)

**2.3 Install `torchattacks`.**
The proposal (Section 3.6) names the `torchattacks` library. This notebook writes its own short FGSM and PGD functions so that every
step can be read and explained, and then **checks them against `torchattacks`** (Section 6.4) to show they behave the same way.

In [ ]:
!pip install -q torchattacks --no-deps

**2.4 Import the libraries.**

In [ ]:
import json
import time
import random
import shutil
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Using device:   ", device)

**2.5 Experiment configuration.**
Every setting of the experiment lives in the next few cells, so it is easy to see (and report) exactly what was run.
They are split into four groups:

1. **Run switches**: what the notebook should do this time.
2. **Training protocol**: shared by *every* model (Proposal Section 3.1: same optimiser, batch size, epochs and seeds).
3. **Training attack**: the adversarial examples used *during training*.
4. **Evaluation attacks**: the attacks used *to test* the finished models. They are kept separate from the training attack on purpose,
   and they are **identical for every defense**. Do not change them between defenses.

**About $\epsilon = 8/255$.** The proposal does not fix a number for $\epsilon$. The original notebook used 0.03; this version uses
$8/255 \approx 0.0314$, the standard CIFAR-10 $L_\infty$ budget in the literature (Madry et al. 2018; Zhang et al. 2019). It is practically the same strength.
The PGD evaluation step size keeps the original rule, step $= \epsilon / 4$ (= 2/255 at $\epsilon = 8/255$), with 20 steps (PGD-20) as before.

In [ ]:
# ---------- 1. Run switches ----------
QUICK_RUN = False              # True = tiny 1-epoch run to test the pipeline (results are meaningless)
OVERWRITE_CHECKPOINTS = False  # True = retrain models even if a finished checkpoint already exists
RE_EVALUATE = False            # True = recompute all evaluation results instead of loading saved ones

In [ ]:
# ---------- 2. Training protocol (shared by every model) ----------
SEEDS = [0]                    # Proposal 3.1 asks for repeated seeds: use [0, 1, 2] for the final experiment
BASELINE_SEED = 100            # seed of the undefended Standard model (also the transfer-attack source)
SOURCE_MODEL_SEEDS = [201, 202]  # seeds of the static source models used ONLY by Ensemble AT
SPLIT_SEED = 42                # fixes the train / validation split
EVAL_SEED = 1234               # fixes the evaluation subsets and the PGD random starts

VALIDATION_SIZE = 5000         # images taken out of the 50,000 training images for validation
TRAIN_SUBSET_SIZE = None       # None = use all remaining 45,000 training images
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LR_DECAY_EPOCHS = [15, 23]     # learning rate is multiplied by LR_DECAY_FACTOR at these epochs (50% and 75% of 30)
LR_DECAY_FACTOR = 0.1
NUM_WORKERS = 2                # background processes that load images

In [ ]:
# ---------- 3. Training attack (used to CREATE training examples) ----------
TRAIN_EPSILON = 8 / 255        # L-infinity budget, the same for all three defenses
TRAIN_PGD_STEP_SIZE = 2 / 255  # step size of the inner attack (SAT's PGD and TRADES' KL attack)
TRAIN_PGD_STEPS = 10           # number of inner attack steps (SAT and TRADES)

TRADES_BETA = 6.0              # TRADES trade-off parameter (beta). Larger = more weight on robustness
RUN_TRADES_BETA_SWEEP = False  # optional extra experiment from the original notebook
TRADES_BETA_SWEEP = [1.0, 3.0, 9.0]

EAT_INCLUDE_CURRENT_MODEL = True  # Ensemble AT: also use the model being trained as one of the sources (Tramer et al. 2018)

In [ ]:
# ---------- 4. Evaluation attacks (used to TEST the finished models; identical for every defense) ----------
EVAL_EPSILON = 8 / 255         # main evaluation budget
EVAL_PGD_STEPS = 20            # PGD-20
EVAL_PGD_STEP_FRACTION = 0.25  # PGD step size = epsilon * 0.25  (2/255 when epsilon = 8/255)
EVAL_EPSILONS_SWEEP = [2 / 255, 4 / 255, 8 / 255, 12 / 255, 16 / 255]  # "multiple perturbation budgets" (Proposal 3.3)

EVAL_NUM_IMAGES = 10000        # test images for the main evaluation (10,000 = the whole CIFAR-10 test set)
SWEEP_NUM_IMAGES = 2000        # fixed test subset for the epsilon sweep (to save time)
MASKING_NUM_IMAGES = 1000      # fixed test subset for the gradient-masking checks
VAL_PGD_NUM_IMAGES = 1000      # validation images attacked after every epoch (for the learning curves)
EVAL_BATCH_SIZE = 256
ECE_NUM_BINS = 15              # number of confidence bins for Expected Calibration Error

**2.6 Quick-run override, method names and folders.**
If `QUICK_RUN = True` this cell shrinks the experiment and switches to a separate folder, so test files never overwrite real ones.

In [ ]:
if QUICK_RUN:
    EPOCHS = 1
    LR_DECAY_EPOCHS = [1]
    TRAIN_SUBSET_SIZE = 2000
    SEEDS = [0]
    EVAL_NUM_IMAGES = 500
    SWEEP_NUM_IMAGES = 200
    MASKING_NUM_IMAGES = 100
    VAL_PGD_NUM_IMAGES = 200
    print("QUICK_RUN is ON: tiny experiment for testing only. Do NOT report these numbers.")

# The three defenses being compared (the independent variable).
METHOD_NAMES = ["SAT", "Ensemble AT", "TRADES"]
RUN_PREFIX = {"SAT": "sat", "Ensemble AT": "eat", "TRADES": "trades"}


def run_name_for(method_name, seed):
    # File name used for a model's checkpoint, e.g. "sat_seed0".
    if method_name == "Standard":
        return "standard_baseline"
    return f"{RUN_PREFIX[method_name]}_seed{seed}"


folder_name = "results_rescnn_quick_test" if QUICK_RUN else "results_rescnn"
RESULTS_DIR = os.path.join(PROJECT_DIR, folder_name)
CHECKPOINT_DIR = os.path.join(RESULTS_DIR, "checkpoints")
RESULTS_FILE = os.path.join(RESULTS_DIR, "evaluation_results.pt")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Results folder:", RESULTS_DIR)
print("Epochs:", EPOCHS, "| Seeds:", SEEDS)

**2.7 Save the configuration.**
The full configuration is written to `experiment_config.json` next to the results, so the experiment can be documented and reproduced.

In [ ]:
experiment_config = {
    "architecture": "ResCNN (Section 4)",
    "dataset": "CIFAR-10",
    "quick_run": QUICK_RUN,
    "seeds": SEEDS,
    "baseline_seed": BASELINE_SEED,
    "source_model_seeds": SOURCE_MODEL_SEEDS,
    "split_seed": SPLIT_SEED,
    "eval_seed": EVAL_SEED,
    "validation_size": VALIDATION_SIZE,
    "train_subset_size": TRAIN_SUBSET_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "SGD",
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "lr_decay_epochs": LR_DECAY_EPOCHS,
    "lr_decay_factor": LR_DECAY_FACTOR,
    "train_epsilon": TRAIN_EPSILON,
    "train_pgd_step_size": TRAIN_PGD_STEP_SIZE,
    "train_pgd_steps": TRAIN_PGD_STEPS,
    "trades_beta": TRADES_BETA,
    "eat_include_current_model": EAT_INCLUDE_CURRENT_MODEL,
    "eval_epsilon": EVAL_EPSILON,
    "eval_pgd_steps": EVAL_PGD_STEPS,
    "eval_pgd_step_fraction": EVAL_PGD_STEP_FRACTION,
    "eval_epsilons_sweep": EVAL_EPSILONS_SWEEP,
    "eval_num_images": EVAL_NUM_IMAGES,
    "sweep_num_images": SWEEP_NUM_IMAGES,
    "masking_num_images": MASKING_NUM_IMAGES,
    "ece_num_bins": ECE_NUM_BINS,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "torch_version": torch.__version__,
}

with open(os.path.join(RESULTS_DIR, "experiment_config.json"), "w") as f:
    json.dump(experiment_config, f, indent=2)

print("Saved experiment_config.json")

**2.8 Control randomness.**
Neural network training uses random numbers (weight initialisation, shuffling, data augmentation, PGD random starts).
`set_seed` fixes all of them so a run can be repeated. We also ask cuDNN (the GPU library) to use deterministic algorithms.
Some GPU operations can still differ very slightly between runs, so tiny differences (a fraction of a percent) are normal.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(SPLIT_SEED)
print("Seeds set.")

---
# 3. Dataset Loading and Preprocessing

**CIFAR-10** has 60,000 colour images of size $32 \times 32$ in 10 balanced classes: 50,000 for training and 10,000 for testing.

We use three disjoint parts:

| Part | Size | Used for |
|---|---|---|
| Training set | 45,000 | Updating the model weights |
| Validation set | 5,000 (taken from the official training images) | Learning curves during training |
| Test set | 10,000 (official test images) | **Only** the final evaluation in Sections 12–16 |

The test set is never used during training, so no evaluation data leaks into training.

**Preprocessing.**
- Images are converted to tensors with pixel values in $[0, 1]$.
- **Training images only** get light data augmentation (random crop with 4-pixel padding and random horizontal flip). Validation and test images are not augmented.
- **Normalisation** (subtracting the CIFAR-10 mean and dividing by the standard deviation) is done *inside the model* (Section 4).
  This matters for the attacks: $\epsilon$ is defined on real pixel values in $[0, 1]$, so the attacks must work on un-normalised images and
  clip the result back into $[0, 1]$. Putting normalisation inside the model means every attack, every defense and `torchattacks`
  all see the same $[0, 1]$ images, and $\epsilon = 8/255$ really means 8 brightness levels.

**3.1 Copy CIFAR-10 to the notebook's own disk.**
Reading images straight from Drive is slow, so (in Colab) one fast local copy is made the first time.
If no copy exists on Drive, `torchvision` downloads CIFAR-10 when the next cell runs.

In [ ]:
DRIVE_DATA_DIR = os.path.join(PROJECT_DIR, "data")

if IN_COLAB:
    DATA_DIR = "/content/data"
    if not os.path.exists(DATA_DIR) and os.path.exists(DRIVE_DATA_DIR):
        shutil.copytree(DRIVE_DATA_DIR, DATA_DIR)
        print("CIFAR-10 copied from Drive to local disk.")
    else:
        print("Using local data folder:", DATA_DIR)
else:
    DATA_DIR = DRIVE_DATA_DIR
    print("Using data folder:", DATA_DIR)

**3.2 Define the image transforms and load the datasets.**
We load the training images twice: once *with* augmentation (for training) and once *without* (for validation).
Both point to the same files; the fixed split in 3.3 decides which image goes where.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),             # (3, 32, 32), values in [0, 1]
])
test_transform = transforms.ToTensor()  # no augmentation, values in [0, 1]

train_set_augmented = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_transform)
train_set_plain = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=test_transform)
test_set = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_transform)

CLASSES = test_set.classes
print("Training images available:", len(train_set_plain))
print("Test images:", len(test_set))
print("Classes:", CLASSES)

**3.3 Make the fixed train / validation split.**
The split is made with `SPLIT_SEED` and saved to `data_split.json`. If the file already exists (for example on a later run or for another seed),
the notebook checks that the split is *exactly* the same, so every defense is trained and validated on identical images.

In [ ]:
split_generator = torch.Generator().manual_seed(SPLIT_SEED)
shuffled_indices = torch.randperm(len(train_set_plain), generator=split_generator).tolist()

validation_indices = shuffled_indices[:VALIDATION_SIZE]
training_indices = shuffled_indices[VALIDATION_SIZE:]
if TRAIN_SUBSET_SIZE is not None:
    training_indices = training_indices[:TRAIN_SUBSET_SIZE]

# The two parts must not share any image.
assert len(set(training_indices) & set(validation_indices)) == 0

split_file = os.path.join(RESULTS_DIR, "data_split.json")
if os.path.exists(split_file):
    with open(split_file) as f:
        saved_split = json.load(f)
    assert saved_split["training_indices"] == training_indices, "Training split differs from the saved split!"
    assert saved_split["validation_indices"] == validation_indices, "Validation split differs from the saved split!"
    print("Split matches the saved split in data_split.json.")
else:
    with open(split_file, "w") as f:
        json.dump({"training_indices": training_indices, "validation_indices": validation_indices}, f)
    print("Saved new split to data_split.json.")

print("Training images:  ", len(training_indices))
print("Validation images:", len(validation_indices))

**3.4 Build the data loaders.**
A `DataLoader` serves the images in mini-batches. Each batch has shape `(BATCH_SIZE, 3, 32, 32)` for images and `(BATCH_SIZE,)` for labels.

The training loader is created by a small function that takes a seed, because the shuffling order is part of what we control:
every defense sees the training images in the same order for the same seed and epoch.

In [ ]:
training_data = Subset(train_set_augmented, training_indices)
validation_data = Subset(train_set_plain, validation_indices)

validation_loader = DataLoader(validation_data, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


def make_train_loader(seed):
    generator = torch.Generator().manual_seed(seed)  # controls the shuffle order and augmentation
    return DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


example_images, example_labels = next(iter(make_train_loader(seed=0)))
print("One training batch:", tuple(example_images.shape), tuple(example_labels.shape))
print("Pixel range:", example_images.min().item(), "to", example_images.max().item())

**3.5 Fix the evaluation images.**
All evaluation in Sections 12–16 uses fixed tensors of test images, so **every model is evaluated on exactly the same images**:

- `X_eval`, `y_eval`: the main evaluation set (`EVAL_NUM_IMAGES` images; the whole test set by default).
- `X_sweep`, `y_sweep`: a fixed subset for the $\epsilon$ sweep.
- `X_mask`, `y_mask`: a fixed subset for the gradient-masking checks.
- `X_val_pgd`, `y_val_pgd`: validation images attacked after every training epoch.

The test images are converted exactly as `ToTensor()` does: shape `(10000, 3, 32, 32)`, values in $[0, 1]$.

In [ ]:
X_test = torch.from_numpy(test_set.data).permute(0, 3, 1, 2).float().div(255)  # (10000, 3, 32, 32)
y_test = torch.tensor(test_set.targets)                                        # (10000,)

eval_generator = torch.Generator().manual_seed(EVAL_SEED)
test_permutation = torch.randperm(len(X_test), generator=eval_generator)

eval_indices = test_permutation[:EVAL_NUM_IMAGES].sort().values
sweep_indices = test_permutation[:SWEEP_NUM_IMAGES].sort().values
mask_indices = test_permutation[:MASKING_NUM_IMAGES].sort().values

X_eval, y_eval = X_test[eval_indices], y_test[eval_indices]
X_sweep, y_sweep = X_test[sweep_indices], y_test[sweep_indices]
X_mask, y_mask = X_test[mask_indices], y_test[mask_indices]

X_val_pgd = torch.stack([validation_data[i][0] for i in range(VAL_PGD_NUM_IMAGES)])
y_val_pgd = torch.tensor([validation_data[i][1] for i in range(VAL_PGD_NUM_IMAGES)])

print("Main evaluation set:", tuple(X_eval.shape))
print("Epsilon-sweep set:  ", tuple(X_sweep.shape))
print("Masking-check set:  ", tuple(X_mask.shape))
print("Validation PGD set: ", tuple(X_val_pgd.shape))

**3.6 Look at the data.**
A quick check that the classes are balanced and the images look right.

In [ ]:
training_labels = np.array(train_set_plain.targets)[training_indices]
validation_labels = np.array(train_set_plain.targets)[validation_indices]
class_counts = pd.DataFrame({
    "train": np.bincount(training_labels, minlength=10),
    "validation": np.bincount(validation_labels, minlength=10),
    "test (eval set)": np.bincount(y_eval.numpy(), minlength=10),
}, index=CLASSES)
display(class_counts)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for class_index in range(10):
    first_image_of_class = (y_test == class_index).nonzero()[0].item()
    axes[class_index].imshow(X_test[first_image_of_class].permute(1, 2, 0).numpy())
    axes[class_index].set_title(CLASSES[class_index], fontsize=9)
    axes[class_index].axis("off")
plt.show()

---
# 4. CNN Architecture: `ResCNN`

The original notebook used a small 4-convolution CNN. Adversarial training needs more model capacity than ordinary training, because the model
has to classify both clean and worst-case perturbed images. We therefore use a **moderately deep residual CNN**.

**What is a residual block?** A normal block computes $y = F(x)$. A residual block computes
$$y = \mathrm{ReLU}\big(F(x) + \mathrm{shortcut}(x)\big)$$
The *skip connection* lets the input go around the convolutions. The block only has to learn the *change* $F(x)$, and gradients can flow
backwards through the shortcut easily, so deeper networks train reliably. When a block changes the number of channels or the image size,
the shortcut uses a $1 \times 1$ convolution so that the two tensors have the same shape and can be added.

**Building blocks used**
- **Batch normalisation (BN)** keeps the activations in a stable range, which makes training faster and more stable.
- **ReLU** is the non-linearity, $\mathrm{ReLU}(z) = \max(0, z)$.
- **Downsampling** is done with stride-2 convolutions (halving the height and width) at the start of stages 2 and 3.
- **Global Average Pooling (GAP)** averages each of the 256 feature maps into one number, so the classifier needs only 256 inputs.
  This uses far fewer parameters than flattening.

**Layer-by-layer shapes** (for a batch of $B$ images)

| Layer | Output shape | Notes |
|---|---|---|
| Input | $(B, 3, 32, 32)$ | pixels in $[0, 1]$ |
| Normalize | $(B, 3, 32, 32)$ | subtract CIFAR mean, divide by std |
| Stem: 3×3 conv + BN + ReLU | $(B, 64, 32, 32)$ | initial feature extraction |
| Stage 1: 2 residual blocks | $(B, 64, 32, 32)$ | 64 channels |
| Stage 2: 2 residual blocks | $(B, 128, 16, 16)$ | first block has stride 2 |
| Stage 3: 2 residual blocks | $(B, 256, 8, 8)$ | first block has stride 2 |
| Global Average Pooling | $(B, 256)$ | |
| Linear | $(B, 10)$ | one logit per class |

That is 13 convolution layers plus 1 linear layer, roughly 2.8 million parameters (the exact number is printed below), compared with about
1.1 million for the old SimpleCNN (its checkpoints were 4.4 MB). It is similar to a ResNet-18 without its last stage, so it is
strong enough for adversarial training on CIFAR-10 while still being trainable on a Colab GPU.

The architecture is defined **once** and every model in the experiment (the baseline, the Ensemble AT source models, SAT, Ensemble AT and TRADES)
is created with the same `create_model()` function.

**4.1 The normalisation layer.**
This tiny layer applies the CIFAR-10 normalisation inside the model (see Section 3 for why).

In [ ]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        # Shape (1, 3, 1, 1) so it broadcasts over (batch, channels, height, width).
        # register_buffer: saved with the model, moved to the GPU with it, but never trained.
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std

**4.2 The residual block.**
Two $3 \times 3$ convolutions, each followed by batch normalisation, plus a shortcut.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        # Main path: conv -> BN -> ReLU -> conv -> BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut path: identity, unless the shape changes (then a 1x1 conv makes the shapes match).
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)   # the skip connection
        return F.relu(out)

**4.3 The full `ResCNN` model.**

In [ ]:
class ResCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.normalize = Normalize(CIFAR_MEAN, CIFAR_STD)

        # Stem: initial feature extraction             -> (B, 64, 32, 32)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        # Three stages of two residual blocks each
        self.stage1 = nn.Sequential(ResidualBlock(64, 64, stride=1),    # -> (B, 64, 32, 32)
                                    ResidualBlock(64, 64, stride=1))
        self.stage2 = nn.Sequential(ResidualBlock(64, 128, stride=2),   # -> (B, 128, 16, 16)
                                    ResidualBlock(128, 128, stride=1))
        self.stage3 = nn.Sequential(ResidualBlock(128, 256, stride=2),  # -> (B, 256, 8, 8)
                                    ResidualBlock(256, 256, stride=1))

        self.global_pool = nn.AdaptiveAvgPool2d(1)                      # -> (B, 256, 1, 1)
        self.classifier = nn.Linear(256, num_classes)                   # -> (B, 10)

    def forward(self, x):
        x = self.normalize(x)
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)     # (B, 256, 1, 1) -> (B, 256)
        return self.classifier(x)   # raw scores (logits), no softmax

**4.4 One function creates every model.**
Every model in this notebook is made by `create_model()`. This guarantees that all defenses use exactly the same architecture.
The cell also checks the output shape and counts the parameters.

In [ ]:
def create_model():
    return ResCNN(num_classes=10).to(device)


test_model = create_model()
dummy_batch = torch.zeros(2, 3, 32, 32, device=device)
print("Output shape for a batch of 2:", tuple(test_model(dummy_batch).shape))   # expected (2, 10)

number_of_parameters = sum(p.numel() for p in test_model.parameters())
print(f"Trainable parameters: {number_of_parameters:,}")
del test_model, dummy_batch

---
# 5. FGSM Implementation

The **Fast Gradient Sign Method** (Goodfellow et al., 2015) takes **one** step in the direction that increases the loss the most:
$$x_{adv} = \mathrm{clip}_{[0,1]}\Big(x + \epsilon \cdot \mathrm{sign}\big(\nabla_x \, \mathcal{L}_{CE}(f(x), y)\big)\Big)$$

- $\nabla_x \mathcal{L}$ is the gradient of the cross-entropy loss with respect to the **input pixels** (not the weights).
- $\mathrm{sign}(\cdot)$ turns it into $+1$ / $-1$ for every pixel, so every pixel moves by exactly $\epsilon$: the perturbation automatically stays inside the $L_\infty$ ball.
- $\mathrm{clip}_{[0,1]}$ keeps the result a valid image.

Two implementation details:
- The model is put in **evaluation mode** while the attack is computed, so batch-normalisation statistics are not changed by the attack. Its previous mode is restored afterwards.
- `torch.autograd.grad` computes the gradient for the images only, so the model's weight gradients are not touched.

FGSM is used in two places: as an **evaluation attack** (Section 13) and as the **training attack of Ensemble AT** (Section 8).

In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    # images: (B, 3, 32, 32) in [0, 1];  labels: (B,)
    was_training = model.training
    model.eval()

    images = images.clone().detach().requires_grad_(True)
    loss = F.cross_entropy(model(images), labels)
    gradient = torch.autograd.grad(loss, images)[0]           # same shape as images

    adversarial_images = images + epsilon * gradient.sign()   # one step of size epsilon
    adversarial_images = torch.clamp(adversarial_images, 0.0, 1.0)

    model.train(was_training)
    return adversarial_images.detach()

---
# 6. PGD Implementation

**Projected Gradient Descent** (Madry et al., 2018) is FGSM repeated many times with a smaller step, and it is the main robustness test in
the proposal (Section 3.3). Starting from a random point inside the $\epsilon$-ball, it repeats:

1. **Step:** $x' = x_{adv} + \alpha \cdot \mathrm{sign}(\nabla_x \mathcal{L}_{CE}(f(x_{adv}), y))$
2. **Project** back into the $\epsilon$-ball around the original image: $\delta = \mathrm{clip}_{[-\epsilon, \epsilon]}(x' - x)$
3. **Clip** to a valid image: $x_{adv} = \mathrm{clip}_{[0,1]}(x + \delta)$

where $\alpha$ is the step size. The random start helps the attack avoid a bad starting point.

**Training attack vs evaluation attack (kept separate on purpose)**

| | Used by | $\epsilon$ | Step size $\alpha$ | Steps | Random start |
|---|---|---|---|---|---|
| Training PGD | SAT (Section 7) | 8/255 | 2/255 | 10 | yes |
| Training KL attack | TRADES (Section 9) | 8/255 | 2/255 | 10 | small Gaussian |
| Training FGSM | Ensemble AT (Section 8) | 8/255 | – | 1 | no |
| **Evaluation FGSM** | all models | 8/255 (+ sweep) | – | 1 | no |
| **Evaluation PGD-20** | all models | 8/255 (+ sweep) | $\epsilon/4$ = 2/255 | 20 | yes |

The evaluation attack is *stronger* than the SAT/TRADES training attack (20 steps instead of 10), which is good practice:
a model should not be tested only with the attack it was trained against.

In [ ]:
def pgd_attack(model, images, labels, epsilon, step_size, num_steps, random_start=True):
    # images: (B, 3, 32, 32) in [0, 1];  labels: (B,)
    was_training = model.training
    model.eval()

    original_images = images.clone().detach()
    adversarial_images = original_images.clone()

    if random_start:
        random_noise = torch.empty_like(adversarial_images).uniform_(-epsilon, epsilon)
        adversarial_images = torch.clamp(adversarial_images + random_noise, 0.0, 1.0)

    for step in range(num_steps):
        adversarial_images.requires_grad_(True)
        loss = F.cross_entropy(model(adversarial_images), labels)
        gradient = torch.autograd.grad(loss, adversarial_images)[0]

        # 1. take a small step uphill on the loss
        adversarial_images = adversarial_images.detach() + step_size * gradient.sign()
        # 2. project back into the epsilon-ball around the original image
        perturbation = torch.clamp(adversarial_images - original_images, -epsilon, epsilon)
        # 3. keep a valid image
        adversarial_images = torch.clamp(original_images + perturbation, 0.0, 1.0)

    model.train(was_training)
    return adversarial_images.detach()

**6.1 A safety check for perturbations.**
This function confirms that an adversarial batch obeys the rules: every pixel changed by at most $\epsilon$, and every pixel is inside $[0, 1]$.
It is called on every attacked batch during evaluation, so a broken attack would stop the notebook instead of silently producing wrong numbers.

In [ ]:
def check_perturbation(clean_images, adversarial_images, epsilon):
    largest_change = (adversarial_images - clean_images).abs().max().item()
    assert largest_change <= epsilon + 1e-6, f"Perturbation {largest_change:.5f} is larger than epsilon {epsilon:.5f}"
    assert adversarial_images.min().item() >= 0.0 and adversarial_images.max().item() <= 1.0, "Pixels outside [0, 1]"

**6.2 Check our attacks against `torchattacks`.**
This is a check of the *code*, not an experiment. We build an **untrained** `ResCNN` and run our FGSM and our PGD (without the random start,
so the result is deterministic) next to the `torchattacks` versions on the same 64 test images. If the two implementations agree,
the maximum pixel difference should be (close to) zero. A tiny difference on a few pixels can happen when a gradient is almost exactly zero.

In [ ]:
import torchattacks

set_seed(0)
check_model = create_model()
check_model.eval()
check_images = X_test[:64].to(device)
check_labels = y_test[:64].to(device)

our_fgsm = fgsm_attack(check_model, check_images, check_labels, EVAL_EPSILON)
library_fgsm = torchattacks.FGSM(check_model, eps=EVAL_EPSILON)(check_images, check_labels)

eval_step_size = EVAL_EPSILON * EVAL_PGD_STEP_FRACTION
our_pgd = pgd_attack(check_model, check_images, check_labels, EVAL_EPSILON, eval_step_size, EVAL_PGD_STEPS, random_start=False)
library_pgd = torchattacks.PGD(check_model, eps=EVAL_EPSILON, alpha=eval_step_size,
                               steps=EVAL_PGD_STEPS, random_start=False)(check_images, check_labels)

print(f"FGSM: max pixel difference = {(our_fgsm - library_fgsm).abs().max().item():.2e}")
print(f"PGD : max pixel difference = {(our_pgd - library_pgd).abs().max().item():.2e}")

check_perturbation(check_images, our_fgsm, EVAL_EPSILON)
check_perturbation(check_images, our_pgd, EVAL_EPSILON)
print("Both attacks stay inside the epsilon-ball and the [0, 1] pixel range.")
del check_model

---
# 7. Standard Adversarial Training (SAT)

Each defense below is written as **one function that computes the training loss for one mini-batch**.
The shared training loop in Section 10 calls this function, then does the usual `backward()` and optimiser step.
So the *only* thing that differs between defenses is this loss function, which keeps the comparison easy to see.

Every loss function returns two things: the loss to minimise, and the logits used to compute the training accuracy shown in the curves.

**7.1 Reference: ordinary (clean) training.**
Used for the undefended **Standard** baseline model and for the static source models of Ensemble AT. It is *not* one of the three defenses.
$$\mathcal{L} = \mathcal{L}_{CE}(f(x), y)$$

In [ ]:
def standard_training_loss(model, images, labels):
    logits = model(images)                    # (B, 10)
    loss = F.cross_entropy(logits, labels)
    return loss, logits

**7.2 SAT (Madry et al., 2018).**
Training is treated as a min-max game: the attacker *maximises* the loss inside the $\epsilon$-ball, and the model *minimises* the loss on what the attacker found:
$$\min_\theta \; \mathbb{E}_{(x,y)} \Big[ \max_{\|\delta\|_\infty \le \epsilon} \mathcal{L}_{CE}\big(f_\theta(x + \delta), y\big) \Big]$$

At **every training step**, PGD-10 ($\epsilon = 8/255$, step $2/255$, random start) is run against the **current** model (Proposal 3.4),
and the model is trained with cross-entropy on these adversarial images only.

*Training accuracy for SAT is measured on the adversarial training images.*

In [ ]:
def sat_loss(model, images, labels):
    # Inner maximisation: PGD against the current model
    adversarial_images = pgd_attack(model, images, labels, epsilon=TRAIN_EPSILON,
                                    step_size=TRAIN_PGD_STEP_SIZE, num_steps=TRAIN_PGD_STEPS, random_start=True)
    # Outer minimisation: cross-entropy on the adversarial images
    logits = model(adversarial_images)
    loss = F.cross_entropy(logits, labels)
    return loss, logits

---
# 8. Ensemble Adversarial Training (Ensemble AT)

**Idea (Tramèr et al., 2018).** In SAT the adversarial examples come from the model being trained. Ensemble AT also uses adversarial examples
made on **other, already-trained, static (frozen) source models** (Proposal 3.4). Because these examples do not depend on the model being trained,
the model learns to resist perturbations that *transfer* from other models. That is the black-box setting Ensemble AT is designed for.
The proposal itself expects Ensemble AT to be **less** resistant to white-box iterative attacks such as PGD.

**How it is implemented here (following Tramèr et al., 2018):**

1. **Source models.** Two independently trained, undefended `ResCNN` models (`SOURCE_MODEL_SEEDS`) are trained *before* Ensemble AT (Section 10)
   and then frozen. They have the same architecture as every other model but different random seeds.
2. **Rotating the source.** For every mini-batch, one source is chosen in turn:
   batch 0 → current model, batch 1 → source model 1, batch 2 → source model 2, batch 3 → current model, …
   (Tramèr et al. alternate between the current model and the static models. Set `EAT_INCLUDE_CURRENT_MODEL = False` to use the static models only.)
3. **Single-step attack.** As in Tramèr et al., the adversarial examples are made with a single-step attack. We use FGSM with the same
   $\epsilon = 8/255$ as the other defenses.
4. **Loss.** Every image is used both clean and adversarial, with equal weight (as in Tramèr et al., following Kurakin et al.):
$$\mathcal{L} = \tfrac{1}{2}\,\mathcal{L}_{CE}(f(x), y) + \tfrac{1}{2}\,\mathcal{L}_{CE}(f(x_{adv}), y)$$
   Putting the clean and adversarial images into one batch and averaging the cross-entropy gives exactly this equal weighting.

**Keeping the transfer evaluation fair.** The transfer attacks in Sections 13–14 are made on the **Standard baseline model**, which is
*not* one of Ensemble AT's source models (it has its own seed, `BASELINE_SEED`). So Ensemble AT is never tested with adversarial images from a model it trained against.

*Training accuracy for Ensemble AT is measured on its adversarial training images.*

In [ ]:
def ensemble_at_loss(model, source_models, images, labels, batch_index):
    # 1. Choose where this batch's adversarial examples come from (rotate through the sources)
    if EAT_INCLUDE_CURRENT_MODEL:
        candidate_sources = [model] + source_models
    else:
        candidate_sources = source_models
    attack_source = candidate_sources[batch_index % len(candidate_sources)]

    # 2. Single-step FGSM adversarial examples made on that source model
    adversarial_images = fgsm_attack(attack_source, images, labels, epsilon=TRAIN_EPSILON)

    # 3. Train on clean + adversarial images with equal weight
    combined_images = torch.cat([images, adversarial_images])   # (2B, 3, 32, 32)
    combined_labels = torch.cat([labels, labels])               # (2B,)
    logits = model(combined_images)                             # (2B, 10)
    loss = F.cross_entropy(logits, combined_labels)

    adversarial_logits = logits[len(images):]                   # second half = adversarial images
    return loss, adversarial_logits

---
# 9. TRADES Implementation

**Idea (Zhang et al., 2019).** TRADES splits the goal into two parts: be **accurate** on clean images, and be **consistent**, so that the prediction does not change
when the image is perturbed. The training loss is
$$\mathcal{L}_{TRADES} = \underbrace{\mathcal{L}_{CE}\big(f(x), y\big)}_{\text{clean accuracy}} \; + \; \beta \cdot \underbrace{\mathrm{KL}\big(f(x) \,\|\, f(x')\big)}_{\text{consistency / robustness}}$$
where $f(\cdot)$ here means the softmax probabilities and $x'$ is the adversarial image.

- **KL divergence** measures how different two probability distributions are (0 means identical).
- **$\beta$** (`TRADES_BETA`) is the trade-off parameter: small $\beta$ favours clean accuracy, large $\beta$ favours robustness (Proposal 3.4).
  The original notebook used $\beta = 6$, the value recommended by Zhang et al. for CIFAR-10.

**Inner maximisation.** Unlike SAT, the TRADES attack does **not** maximise the cross-entropy with the true label. It maximises the KL divergence
between the clean prediction and the perturbed prediction:
$$x' = \arg\max_{\|x' - x\|_\infty \le \epsilon} \mathrm{KL}\big(f(x) \,\|\, f(x')\big)$$
It is solved with the same PGD-style steps (10 steps of $2/255$, $\epsilon = 8/255$, the same budget as SAT), starting from the clean image plus very small
Gaussian noise ($0.001 \cdot \mathcal{N}(0, 1)$), as in the official TRADES code.

In PyTorch, `F.kl_div(log_q, p)` computes $\mathrm{KL}(p \,\|\, q)$, so we pass the **log**-probabilities of the adversarial image and the probabilities of the clean image.

**9.1 The TRADES inner attack (KL maximisation).**

In [ ]:
def trades_attack(model, images, epsilon, step_size, num_steps):
    # images: (B, 3, 32, 32) in [0, 1]. Note: no labels needed, the target is the model's own clean prediction.
    was_training = model.training
    model.eval()

    original_images = images.clone().detach()
    with torch.no_grad():
        clean_probabilities = F.softmax(model(original_images), dim=1)   # (B, 10), fixed target

    adversarial_images = original_images + 0.001 * torch.randn_like(original_images)
    adversarial_images = torch.clamp(adversarial_images, 0.0, 1.0)

    for step in range(num_steps):
        adversarial_images.requires_grad_(True)
        adversarial_log_probabilities = F.log_softmax(model(adversarial_images), dim=1)
        kl_loss = F.kl_div(adversarial_log_probabilities, clean_probabilities, reduction="sum")
        gradient = torch.autograd.grad(kl_loss, adversarial_images)[0]

        adversarial_images = adversarial_images.detach() + step_size * gradient.sign()
        perturbation = torch.clamp(adversarial_images - original_images, -epsilon, epsilon)
        adversarial_images = torch.clamp(original_images + perturbation, 0.0, 1.0)

    model.train(was_training)
    return adversarial_images.detach()

**9.2 The TRADES loss.**
The clean cross-entropy plus $\beta$ times the KL term. `reduction="batchmean"` divides the summed KL by the batch size, as in the TRADES paper.
Both the clean and the adversarial predictions receive gradients, again as in the official implementation.

*Training accuracy for TRADES is measured on its adversarial training images.*

In [ ]:
def trades_loss(model, images, labels, beta):
    adversarial_images = trades_attack(model, images, epsilon=TRAIN_EPSILON,
                                       step_size=TRAIN_PGD_STEP_SIZE, num_steps=TRAIN_PGD_STEPS)

    clean_logits = model(images)                 # (B, 10)
    adversarial_logits = model(adversarial_images)  # (B, 10)

    natural_loss = F.cross_entropy(clean_logits, labels)
    robust_loss = F.kl_div(F.log_softmax(adversarial_logits, dim=1),
                           F.softmax(clean_logits, dim=1),
                           reduction="batchmean")
    loss = natural_loss + beta * robust_loss
    return loss, adversarial_logits

---
# 10. Model Training

All models are trained with **one shared training loop** and **one shared configuration** (Section 2.5):

| Setting | Value |
|---|---|
| Optimiser | SGD, momentum 0.9, weight decay $5 \times 10^{-4}$ |
| Learning rate | 0.1, multiplied by 0.1 at 50% and 75% of training |
| Batch size | 128 |
| Epochs | `EPOCHS` (30 by default), the same for every model |
| Seeds | `SEEDS`, the same for every defense |

**Models trained in this section**

| Model | Training | Role |
|---|---|---|
| Standard baseline | clean training, seed `BASELINE_SEED` | undefended reference **and** source of the transfer attacks |
| Source models 1 and 2 | clean training, seeds `SOURCE_MODEL_SEEDS` | static source models for Ensemble AT only |
| SAT | Section 7 | defense 1 |
| Ensemble AT | Section 8 | defense 2 |
| TRADES | Section 9 | defense 3 |

**Same seed, same conditions.** At the start of every epoch the random seed is reset to `seed * 1000 + epoch`, and the shuffle order comes from that seed.
So for the same seed, SAT, Ensemble AT and TRADES start from **the same initial weights** and see **the same images in the same order with the same augmentation**.
This also means a run that resumes after a disconnection continues exactly as if it had not stopped (apart from tiny GPU differences).

**Time.** Adversarial training is slow, because every training step also runs the attack. SAT and TRADES run 10 attack steps per batch,
so expect *several hours per model* on a Colab GPU. Ensemble AT and clean training are much faster. **Training time is one of the measured
results (Proposal Objective 3)**: it is recorded per epoch, counts only the training passes (not validation) and is saved in each checkpoint.
Training times are only comparable when all models are trained on the same type of GPU, which is why the GPU name is saved too.

**10.1 Choose the right loss function for each method.**

In [ ]:
def compute_method_loss(method_name, model, images, labels, batch_index, source_models, trades_beta):
    if method_name == "Standard":
        return standard_training_loss(model, images, labels)
    elif method_name == "SAT":
        return sat_loss(model, images, labels)
    elif method_name == "Ensemble AT":
        return ensemble_at_loss(model, source_models, images, labels, batch_index)
    elif method_name == "TRADES":
        return trades_loss(model, images, labels, trades_beta)
    else:
        raise ValueError(f"Unknown method: {method_name}")

**10.2 Train for one epoch.**
One pass over the training set. For every batch: compute the method's loss, compute the gradients, update the weights.

In [ ]:
def train_one_epoch(model, optimizer, train_loader, method_name, source_models, trades_beta):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for batch_index, (images, labels) in enumerate(train_loader):
        images = images.to(device)   # (B, 3, 32, 32)
        labels = labels.to(device)   # (B,)

        optimizer.zero_grad()
        loss, logits = compute_method_loss(method_name, model, images, labels,
                                           batch_index, source_models, trades_beta)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_images += labels.size(0)

    average_loss = total_loss / total_images
    accuracy = 100.0 * total_correct / total_images
    return average_loss, accuracy

**10.3 Measure the model on the validation set.**
After every epoch we measure, on validation images that are never used for training:
- clean cross-entropy loss and accuracy (all 5,000 validation images),
- PGD-20 cross-entropy loss and accuracy (the first `VAL_PGD_NUM_IMAGES` validation images, using the evaluation PGD settings).

Unlike the training losses (which are different formulas for different methods), the validation losses are the **same cross-entropy for every method**,
so the validation curves can be compared directly.

In [ ]:
def evaluate_on_validation(model):
    model.eval()

    # Clean validation loss and accuracy
    clean_loss_sum, clean_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            clean_loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
            clean_correct += (logits.argmax(dim=1) == labels).sum().item()

    # PGD validation loss and accuracy (on a fixed validation subset)
    pgd_loss_sum, pgd_correct = 0.0, 0
    for start in range(0, len(X_val_pgd), EVAL_BATCH_SIZE):
        images = X_val_pgd[start:start + EVAL_BATCH_SIZE].to(device)
        labels = y_val_pgd[start:start + EVAL_BATCH_SIZE].to(device)
        adversarial_images = pgd_attack(model, images, labels, EVAL_EPSILON,
                                        EVAL_EPSILON * EVAL_PGD_STEP_FRACTION, EVAL_PGD_STEPS)
        with torch.no_grad():
            logits = model(adversarial_images)
            pgd_loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
            pgd_correct += (logits.argmax(dim=1) == labels).sum().item()

    return {
        "val_clean_loss": clean_loss_sum / len(validation_data),
        "val_clean_acc": 100.0 * clean_correct / len(validation_data),
        "val_pgd_loss": pgd_loss_sum / len(X_val_pgd),
        "val_pgd_acc": 100.0 * pgd_correct / len(X_val_pgd),
    }

**10.4 Saving and loading checkpoints.**
A *checkpoint* is a saved model (a `.pth` file). Two kinds are written to `CHECKPOINT_DIR` on Drive:

- `<run>_in_progress.pth`, overwritten after **every epoch**. It holds the model, optimiser, scheduler, epoch number and learning curves, so training can resume after a disconnection.
- `<run>.pth`, written once when training is **finished**. Section 11 loads these for evaluation. The in-progress file is then deleted.

For every method we use the model **after the final epoch** (the same rule for every defense, so checkpoint choice cannot favour one method).

(`weights_only=False` is needed when loading because the checkpoint also stores the learning-rate scheduler and the history. Only load checkpoints you created yourself.)

In [ ]:
def checkpoint_path(run_name):
    return os.path.join(CHECKPOINT_DIR, f"{run_name}.pth")


def in_progress_path(run_name):
    return os.path.join(CHECKPOINT_DIR, f"{run_name}_in_progress.pth")


def save_checkpoint(path, model, optimizer, scheduler, epoch, history, run_info):
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "epoch": epoch,
        "history": history,
        "run_info": run_info,
    }, path)


def load_trained_model(run_name):
    checkpoint = torch.load(checkpoint_path(run_name), map_location=device, weights_only=False)
    model = create_model()
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model, checkpoint["history"], checkpoint["run_info"]

**10.5 Train one model from start to finish.**
`train_defense` trains one model for one seed. It:
1. skips training if a finished checkpoint already exists (unless `OVERWRITE_CHECKPOINTS = True`),
2. otherwise builds a fresh `ResCNN`, the optimiser and the learning-rate schedule,
3. resumes from an in-progress checkpoint if there is one,
4. runs the epochs, recording the learning curves and the training time, and saving after every epoch.

In [ ]:
def train_defense(method_name, seed, run_name, source_models=None, trades_beta=TRADES_BETA):
    if os.path.exists(checkpoint_path(run_name)) and not OVERWRITE_CHECKPOINTS:
        print(f"[{run_name}] finished checkpoint found, skipping training.")
        return

    set_seed(seed)   # same seed -> same initial weights for every method
    model = create_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=LR_DECAY_EPOCHS, gamma=LR_DECAY_FACTOR)

    history = {"epoch": [], "train_loss": [], "train_acc": [], "val_clean_loss": [], "val_clean_acc": [],
               "val_pgd_loss": [], "val_pgd_acc": [], "epoch_seconds": [], "learning_rate": []}
    run_info = {"method": method_name, "seed": seed, "run_name": run_name, "epochs": EPOCHS,
                "trades_beta": trades_beta if method_name == "TRADES" else None,
                "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
    start_epoch = 1

    if os.path.exists(in_progress_path(run_name)) and not OVERWRITE_CHECKPOINTS:
        checkpoint = torch.load(in_progress_path(run_name), map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        history = checkpoint["history"]
        start_epoch = checkpoint["epoch"] + 1
        print(f"[{run_name}] resuming from epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS + 1):
        set_seed(seed * 1000 + epoch)   # reproducible attacks / augmentation, also after resuming
        train_loader = make_train_loader(seed * 1000 + epoch)
        current_lr = optimizer.param_groups[0]["lr"]

        start_time = time.time()
        train_loss, train_acc = train_one_epoch(model, optimizer, train_loader, method_name, source_models, trades_beta)
        epoch_seconds = time.time() - start_time   # training time only (validation excluded)
        scheduler.step()

        validation = evaluate_on_validation(model)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        for key, value in validation.items():
            history[key].append(value)
        history["epoch_seconds"].append(epoch_seconds)
        history["learning_rate"].append(current_lr)

        print(f"[{run_name}] epoch {epoch:2d}/{EPOCHS} | lr {current_lr:.4f} | "
              f"train loss {train_loss:.3f} acc {train_acc:5.1f}% | "
              f"val clean {validation['val_clean_acc']:5.1f}% | val PGD {validation['val_pgd_acc']:5.1f}% | "
              f"{epoch_seconds / 60:.1f} min")
        save_checkpoint(in_progress_path(run_name), model, optimizer, scheduler, epoch, history, run_info)

    run_info["total_training_hours"] = sum(history["epoch_seconds"]) / 3600
    save_checkpoint(checkpoint_path(run_name), model, optimizer, scheduler, EPOCHS, history, run_info)
    if os.path.exists(in_progress_path(run_name)):
        os.remove(in_progress_path(run_name))
    print(f"[{run_name}] finished in {run_info['total_training_hours']:.2f} h. Saved {checkpoint_path(run_name)}")

**10.6 Safety backup before overwriting.**
If you set `OVERWRITE_CHECKPOINTS = True`, this cell first copies the whole checkpoint folder to a dated backup folder so old models are not lost.
With `OVERWRITE_CHECKPOINTS = False` it does nothing.

In [ ]:
if OVERWRITE_CHECKPOINTS and len(os.listdir(CHECKPOINT_DIR)) > 0:
    backup_dir = CHECKPOINT_DIR + "_backup_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    shutil.copytree(CHECKPOINT_DIR, backup_dir)
    print("Backed up checkpoints to", backup_dir)
else:
    print("Nothing to back up.")

**10.7 Train the Standard baseline and the Ensemble AT source models.**
These use ordinary clean training (Section 7.1). They must be trained first because Ensemble AT needs the source models and the transfer evaluation needs the baseline.

In [ ]:
train_defense("Standard", BASELINE_SEED, run_name_for("Standard", BASELINE_SEED))

for source_number, source_seed in enumerate(SOURCE_MODEL_SEEDS, start=1):
    train_defense("Standard", source_seed, f"eat_source_model_{source_number}")

**10.8 Train Defense 1: SAT.**

In [ ]:
for seed in SEEDS:
    train_defense("SAT", seed, run_name_for("SAT", seed))

**10.9 Train Defense 2: Ensemble AT.**
First the static source models are loaded and **frozen** (their weights can no longer change and they stay in evaluation mode).

In [ ]:
source_models = []
for source_number in range(1, len(SOURCE_MODEL_SEEDS) + 1):
    source_model, _, _ = load_trained_model(f"eat_source_model_{source_number}")
    for parameter in source_model.parameters():
        parameter.requires_grad_(False)
    source_models.append(source_model)
print("Loaded", len(source_models), "frozen source models.")

for seed in SEEDS:
    train_defense("Ensemble AT", seed, run_name_for("Ensemble AT", seed), source_models=source_models)

**10.10 Train Defense 3: TRADES** (with $\beta$ = `TRADES_BETA`).

In [ ]:
for seed in SEEDS:
    train_defense("TRADES", seed, run_name_for("TRADES", seed), trades_beta=TRADES_BETA)

**10.11 Optional: TRADES $\beta$ sweep.**
The original notebook also trained TRADES with $\beta \in \{1, 3, 9\}$ to show the accuracy/robustness trade-off that $\beta$ controls.
This is a *secondary* experiment (it varies $\beta$, not the defense). It only runs when `RUN_TRADES_BETA_SWEEP = True`, uses the first seed only, and is evaluated in Section 16.10.

In [ ]:
if RUN_TRADES_BETA_SWEEP:
    for beta in TRADES_BETA_SWEEP:
        train_defense("TRADES", SEEDS[0], f"trades_beta{beta}_seed{SEEDS[0]}", trades_beta=beta)
else:
    print("Skipped (RUN_TRADES_BETA_SWEEP = False).")

---
# 11. Model Checkpointing: Check and Load the Trained Models

From here on, the notebook only *uses* the saved models. If training is finished, you can start a new Colab session,
run Sections 2–9 (fast, they only define things) and jump straight here.

**11.1 List the checkpoints and check that every required model is present.**

In [ ]:
required_run_names = [run_name_for("Standard", BASELINE_SEED)]
required_run_names += [f"eat_source_model_{n}" for n in range(1, len(SOURCE_MODEL_SEEDS) + 1)]
required_run_names += [run_name_for(method, seed) for method in METHOD_NAMES for seed in SEEDS]

for file_name in sorted(os.listdir(CHECKPOINT_DIR)):
    size_mb = os.path.getsize(os.path.join(CHECKPOINT_DIR, file_name)) / 1e6
    print(f"  {file_name}   ({size_mb:.1f} MB)")

missing = [name for name in required_run_names if not os.path.exists(checkpoint_path(name))]
print("\nAll required models are present." if not missing else f"\nMISSING (train them in Section 10): {missing}")

**11.2 Load the models to be evaluated.**
Each model is stored under a key `(method, seed)`, for example `("SAT", 0)`. The Standard baseline is included as an undefended reference point;
it is not one of the three defenses being compared.

- `evaluation_runs`: every model to evaluate (the baseline, plus every defense × seed).
- `primary_runs`: the baseline plus the first seed of each defense, used for the detailed per-model analyses (confusion matrices, reports, plots).

In [ ]:
evaluation_runs = [("Standard", BASELINE_SEED)] + [(method, seed) for method in METHOD_NAMES for seed in SEEDS]
primary_seed = SEEDS[0]
primary_runs = [("Standard", BASELINE_SEED)] + [(method, primary_seed) for method in METHOD_NAMES]

trained_models = {}
training_histories = {}
run_infos = {}
for run_key in evaluation_runs:
    method, seed = run_key
    model, history, run_info = load_trained_model(run_name_for(method, seed))
    trained_models[run_key] = model
    training_histories[run_key] = history
    run_infos[run_key] = run_info
    print(f"Loaded {run_key}: trained {len(history['epoch'])} epochs, "
          f"{run_info['total_training_hours']:.2f} h on {run_info['device']}")


def run_label(run_key):
    # Readable name for tables and plots
    method, seed = run_key
    if method == "Standard":
        return "Standard (undefended)"
    if len(SEEDS) > 1:
        return f"{method} (seed {seed})"
    return method

**11.3 Fairness check: identical architecture.**
This confirms that every loaded model is a `ResCNN` with exactly the same layers and number of parameters.

In [ ]:
reference_model = trained_models[evaluation_runs[0]]
reference_shapes = [tuple(p.shape) for p in reference_model.state_dict().values()]

for run_key, model in trained_models.items():
    assert isinstance(model, ResCNN)
    assert [tuple(p.shape) for p in model.state_dict().values()] == reference_shapes
    print(f"{run_label(run_key):25s} ResCNN, {sum(p.numel() for p in model.parameters()):,} parameters")
print("\nAll models share the identical architecture.")

---
# 12. Clean Evaluation

**How evaluation works.** `get_predictions` runs one model over a fixed set of images in batches. If an attack is requested it first attacks each batch
**against that same model** (white-box), checks that the perturbation is valid, and then records for every image:
the prediction, the maximum softmax probability (confidence), the full probability vector and the cross-entropy loss.
Everything else in Sections 12–16 is computed from these saved predictions.

Before each model's evaluation the random seed is reset to `EVAL_SEED`, so PGD's random start uses the same noise for every model.

**Saved results.** All predictions are saved to `evaluation_results.pt` after each section. If that file exists and `RE_EVALUATE = False`,
finished parts are loaded instead of recomputed. If you retrain any model, set `RE_EVALUATE = True` so the old results are not reused.

**12.1 The prediction function and two small helpers.**

In [ ]:
def get_predictions(model, images, labels, attack="none", epsilon=EVAL_EPSILON, pgd_steps=EVAL_PGD_STEPS):
    # images: (N, 3, 32, 32) in [0, 1] on the CPU;  labels: (N,)
    # attack: "none", "fgsm" or "pgd" (white-box, computed against `model` itself)
    model.eval()
    predictions, confidences, probabilities, losses = [], [], [], []

    for start in range(0, len(images), EVAL_BATCH_SIZE):
        batch_images = images[start:start + EVAL_BATCH_SIZE].to(device)
        batch_labels = labels[start:start + EVAL_BATCH_SIZE].to(device)

        if attack == "fgsm":
            attacked = fgsm_attack(model, batch_images, batch_labels, epsilon)
            check_perturbation(batch_images, attacked, epsilon)
            batch_images = attacked
        elif attack == "pgd":
            attacked = pgd_attack(model, batch_images, batch_labels, epsilon,
                                  step_size=epsilon * EVAL_PGD_STEP_FRACTION, num_steps=pgd_steps, random_start=True)
            check_perturbation(batch_images, attacked, epsilon)
            batch_images = attacked
        elif attack != "none":
            raise ValueError(f"Unknown attack: {attack}")

        with torch.no_grad():
            logits = model(batch_images)                       # (B, 10)
            batch_probabilities = F.softmax(logits, dim=1)     # (B, 10)
            batch_losses = F.cross_entropy(logits, batch_labels, reduction="none")  # (B,)
        batch_confidences, batch_predictions = batch_probabilities.max(dim=1)

        predictions.append(batch_predictions.cpu())
        confidences.append(batch_confidences.cpu())
        probabilities.append(batch_probabilities.cpu())
        losses.append(batch_losses.cpu())

    return {
        "labels": labels.numpy(),
        "predictions": torch.cat(predictions).numpy(),
        "confidences": torch.cat(confidences).numpy(),
        "probabilities": torch.cat(probabilities).numpy(),
        "losses": torch.cat(losses).numpy(),
    }


def accuracy_percent(result):
    return 100.0 * np.mean(result["predictions"] == result["labels"])


def mean_loss(result):
    return float(np.mean(result["losses"]))

**12.2 Load previously saved results (if any).**

In [ ]:
if os.path.exists(RESULTS_FILE) and not RE_EVALUATE:
    results = torch.load(RESULTS_FILE, weights_only=False)
    print("Loaded saved evaluation results from", RESULTS_FILE)
else:
    results = {}
    print("Starting a fresh evaluation.")

for run_key in evaluation_runs:
    results.setdefault(run_key, {})


def save_results():
    torch.save(results, RESULTS_FILE)

**12.3 Evaluate every model on the clean test images.**

In [ ]:
for run_key in evaluation_runs:
    if "clean" in results[run_key]:
        continue
    torch.manual_seed(EVAL_SEED)
    results[run_key]["clean"] = get_predictions(trained_models[run_key], X_eval, y_eval, attack="none")
    print(f"{run_label(run_key):25s} clean evaluation done")
save_results()

**12.4 Clean results.**

In [ ]:
clean_rows = []
for run_key in evaluation_runs:
    clean_rows.append({
        "model": run_label(run_key),
        "clean accuracy (%)": accuracy_percent(results[run_key]["clean"]),
        "clean loss": mean_loss(results[run_key]["clean"]),
    })
clean_table = pd.DataFrame(clean_rows).set_index("model")
display(clean_table.round(3))

---
# 13. FGSM Robustness Evaluation

Every model is tested with FGSM at $\epsilon = 8/255$ in two settings (Proposal 3.3):

- **White-box:** the FGSM images are made on the model being tested.
- **Transfer (black-box):** the FGSM images are made **once** on the Standard baseline model and then shown to every defense.
  All three defenses receive *exactly the same* transfer images. (The baseline itself is not given a transfer score, because for it this would be a white-box attack.)

**Attack Success Rate (ASR)** is calculated over the images that the model classified **correctly when clean**:
$$\mathrm{ASR} = \frac{\#\{\text{correct when clean, wrong after the attack}\}}{\#\{\text{correct when clean}\}} \times 100\%$$
This measures how often the attack *turns a correct answer into a wrong one*, separately from the model's clean accuracy.

**13.1 Helper functions: ASR and transfer images.**

In [ ]:
def attack_success_rate(clean_result, adversarial_result):
    clean_correct = clean_result["predictions"] == clean_result["labels"]
    adversarial_wrong = adversarial_result["predictions"] != adversarial_result["labels"]
    return 100.0 * np.sum(clean_correct & adversarial_wrong) / np.sum(clean_correct)


def make_transfer_images(source_model, images, labels, attack, epsilon=EVAL_EPSILON):
    # Creates adversarial images on `source_model`; they are later shown to OTHER models.
    source_model.eval()
    adversarial_batches = []
    for start in range(0, len(images), EVAL_BATCH_SIZE):
        batch_images = images[start:start + EVAL_BATCH_SIZE].to(device)
        batch_labels = labels[start:start + EVAL_BATCH_SIZE].to(device)
        if attack == "fgsm":
            attacked = fgsm_attack(source_model, batch_images, batch_labels, epsilon)
        else:
            attacked = pgd_attack(source_model, batch_images, batch_labels, epsilon,
                                  step_size=epsilon * EVAL_PGD_STEP_FRACTION, num_steps=EVAL_PGD_STEPS)
        check_perturbation(batch_images, attacked, epsilon)
        adversarial_batches.append(attacked.cpu())
    return torch.cat(adversarial_batches)   # (N, 3, 32, 32)


baseline_key = ("Standard", BASELINE_SEED)
defense_runs = [run_key for run_key in evaluation_runs if run_key != baseline_key]

**13.2 White-box FGSM.**

In [ ]:
for run_key in evaluation_runs:
    if "fgsm" in results[run_key]:
        continue
    torch.manual_seed(EVAL_SEED)
    results[run_key]["fgsm"] = get_predictions(trained_models[run_key], X_eval, y_eval, attack="fgsm")
    print(f"{run_label(run_key):25s} white-box FGSM done")
save_results()

**13.3 Transfer FGSM (made on the Standard baseline).**

In [ ]:
if any("fgsm_transfer" not in results[run_key] for run_key in defense_runs):
    fgsm_transfer_images = make_transfer_images(trained_models[baseline_key], X_eval, y_eval, attack="fgsm")
    for run_key in defense_runs:
        # The transfer images are given to the defense as ordinary inputs (no further attack).
        results[run_key]["fgsm_transfer"] = get_predictions(trained_models[run_key], fgsm_transfer_images, y_eval, attack="none")
        print(f"{run_label(run_key):25s} transfer FGSM done")
    save_results()
else:
    print("Transfer FGSM results already available.")

**13.4 FGSM results** ($\epsilon = 8/255$).

In [ ]:
fgsm_rows = []
for run_key in evaluation_runs:
    run_results = results[run_key]
    row = {
        "model": run_label(run_key),
        "clean acc (%)": accuracy_percent(run_results["clean"]),
        "FGSM white-box acc (%)": accuracy_percent(run_results["fgsm"]),
        "FGSM white-box loss": mean_loss(run_results["fgsm"]),
        "FGSM white-box ASR (%)": attack_success_rate(run_results["clean"], run_results["fgsm"]),
    }
    if "fgsm_transfer" in run_results:
        row["FGSM transfer acc (%)"] = accuracy_percent(run_results["fgsm_transfer"])
        row["FGSM transfer ASR (%)"] = attack_success_rate(run_results["clean"], run_results["fgsm_transfer"])
    fgsm_rows.append(row)
fgsm_table = pd.DataFrame(fgsm_rows).set_index("model")
display(fgsm_table.round(3))

---
# 14. PGD Robustness Evaluation

PGD-20 ($\epsilon = 8/255$, step $2/255$, random start) is the **primary robustness test** (Proposal 3.3). As for FGSM, it is run in the white-box
setting and in the transfer setting (PGD-20 images made once on the Standard baseline, identical for every defense).

**14.1 White-box PGD-20.**

In [ ]:
for run_key in evaluation_runs:
    if "pgd" in results[run_key]:
        continue
    torch.manual_seed(EVAL_SEED)
    results[run_key]["pgd"] = get_predictions(trained_models[run_key], X_eval, y_eval, attack="pgd")
    print(f"{run_label(run_key):25s} white-box PGD-{EVAL_PGD_STEPS} done")
save_results()

**14.2 Transfer PGD-20 (made on the Standard baseline).**

In [ ]:
if any("pgd_transfer" not in results[run_key] for run_key in defense_runs):
    torch.manual_seed(EVAL_SEED)
    pgd_transfer_images = make_transfer_images(trained_models[baseline_key], X_eval, y_eval, attack="pgd")
    for run_key in defense_runs:
        results[run_key]["pgd_transfer"] = get_predictions(trained_models[run_key], pgd_transfer_images, y_eval, attack="none")
        print(f"{run_label(run_key):25s} transfer PGD done")
    save_results()
else:
    print("Transfer PGD results already available.")

**14.3 PGD results** ($\epsilon = 8/255$).

In [ ]:
pgd_rows = []
for run_key in evaluation_runs:
    run_results = results[run_key]
    row = {
        "model": run_label(run_key),
        "clean acc (%)": accuracy_percent(run_results["clean"]),
        f"PGD-{EVAL_PGD_STEPS} white-box acc (%)": accuracy_percent(run_results["pgd"]),
        f"PGD-{EVAL_PGD_STEPS} white-box loss": mean_loss(run_results["pgd"]),
        f"PGD-{EVAL_PGD_STEPS} white-box ASR (%)": attack_success_rate(run_results["clean"], run_results["pgd"]),
    }
    if "pgd_transfer" in run_results:
        row[f"PGD-{EVAL_PGD_STEPS} transfer acc (%)"] = accuracy_percent(run_results["pgd_transfer"])
        row[f"PGD-{EVAL_PGD_STEPS} transfer ASR (%)"] = attack_success_rate(run_results["clean"], run_results["pgd_transfer"])
    pgd_rows.append(row)
pgd_table = pd.DataFrame(pgd_rows).set_index("model")
display(pgd_table.round(3))

**14.4 Robustness at several perturbation budgets.**
The proposal asks for attacks at multiple budgets $\epsilon$ (Section 3.3). Every model is attacked with white-box FGSM and PGD-20 at each $\epsilon$ in
`EVAL_EPSILONS_SWEEP` (2, 4, 8, 12 and 16 out of 255) on the same fixed subset `X_sweep`. The PGD step size follows the same rule, $\epsilon/4$, at every budget.

In [ ]:
for run_key in evaluation_runs:
    sweep = results[run_key].setdefault("sweep", {})
    for epsilon in EVAL_EPSILONS_SWEEP:
        for attack in ["fgsm", "pgd"]:
            sweep_key = f"{attack}_{round(epsilon * 255)}"   # e.g. "pgd_8" means PGD at 8/255
            if sweep_key in sweep:
                continue
            torch.manual_seed(EVAL_SEED)
            prediction = get_predictions(trained_models[run_key], X_sweep, y_sweep, attack=attack, epsilon=epsilon)
            sweep[sweep_key] = accuracy_percent(prediction)
    print(f"{run_label(run_key):25s} epsilon sweep done")
save_results()

sweep_rows = []
for run_key in evaluation_runs:
    row = {"model": run_label(run_key)}
    for epsilon in EVAL_EPSILONS_SWEEP:
        for attack in ["fgsm", "pgd"]:
            row[f"{attack.upper()} {round(epsilon * 255)}/255"] = results[run_key]["sweep"][f"{attack}_{round(epsilon * 255)}"]
    sweep_rows.append(row)
display(pd.DataFrame(sweep_rows).set_index("model").round(2))

**14.5 Gradient-masking checks.**
A defense can look robust only because it makes the gradients useless for the attacker (*gradient masking*, also called obfuscated gradients),
not because it is truly robust. The proposal (Section 3.5) requires a check for this. We run five standard warning signs for every model:

1. **Iterative ≥ single-step:** PGD-20 accuracy should not be higher than FGSM accuracy.
2. **White-box ≥ transfer:** white-box PGD (using the model's own gradients) should be at least as strong as transfer PGD.
3. **Monotonic in $\epsilon$:** accuracy should not go *up* when $\epsilon$ gets bigger.
4. **Large $\epsilon$ breaks every model:** at $\epsilon = 64/255$ PGD can change images a lot, so accuracy should be close to 0%.
5. **More steps do not help much:** PGD-100 should not be much stronger than PGD-20 (if it is, PGD-20 under-estimates the attack).

A `CHECK` does not prove gradient masking, but it means the robust accuracy of that model must be interpreted with care.
Checks 4 and 5 use the fixed subset `X_mask`.

In [ ]:
LARGE_EPSILON = 64 / 255
TOLERANCE = 1.0   # percentage points of slack for random variation

for run_key in evaluation_runs:
    masking = results[run_key].setdefault("masking", {})
    model = trained_models[run_key]
    if "pgd20" not in masking:
        torch.manual_seed(EVAL_SEED)
        masking["pgd20"] = accuracy_percent(get_predictions(model, X_mask, y_mask, attack="pgd", pgd_steps=20))
        torch.manual_seed(EVAL_SEED)
        masking["pgd100"] = accuracy_percent(get_predictions(model, X_mask, y_mask, attack="pgd", pgd_steps=100))
        torch.manual_seed(EVAL_SEED)
        masking["large_eps"] = accuracy_percent(get_predictions(model, X_mask, y_mask, attack="pgd",
                                                                epsilon=LARGE_EPSILON, pgd_steps=20))
save_results()

In [ ]:
def pass_or_check(condition):
    return "PASS" if condition else "CHECK"


masking_rows = []
for run_key in evaluation_runs:
    run_results = results[run_key]
    fgsm_acc = accuracy_percent(run_results["fgsm"])
    pgd_acc = accuracy_percent(run_results["pgd"])

    fgsm_curve = [run_results["sweep"][f"fgsm_{round(e * 255)}"] for e in EVAL_EPSILONS_SWEEP]
    pgd_curve = [run_results["sweep"][f"pgd_{round(e * 255)}"] for e in EVAL_EPSILONS_SWEEP]
    monotonic = True
    for curve in [fgsm_curve, pgd_curve]:
        for i in range(1, len(curve)):
            if curve[i] > curve[i - 1] + TOLERANCE:
                monotonic = False

    row = {
        "model": run_label(run_key),
        "1. PGD <= FGSM": pass_or_check(pgd_acc <= fgsm_acc + TOLERANCE),
        "3. monotonic in eps": pass_or_check(monotonic),
        "4. eps=64/255 acc (%)": run_results["masking"]["large_eps"],
        "4. large eps check": pass_or_check(run_results["masking"]["large_eps"] < 10.0),
        "5. PGD-20 acc (%)": run_results["masking"]["pgd20"],
        "5. PGD-100 acc (%)": run_results["masking"]["pgd100"],
        "5. steps check": pass_or_check(run_results["masking"]["pgd20"] - run_results["masking"]["pgd100"] <= 2.0),
    }
    if "pgd_transfer" in run_results:
        row["2. white-box <= transfer"] = pass_or_check(pgd_acc <= accuracy_percent(run_results["pgd_transfer"]) + TOLERANCE)
    else:
        row["2. white-box <= transfer"] = "n/a"
    masking_rows.append(row)

masking_table = pd.DataFrame(masking_rows).set_index("model")
display(masking_table.round(2))

---
# 15. Confidence and Calibration Analysis

**Confidence** here means the model's highest softmax probability for an image, $\max_k p_k(x)$. For example, "cat, 92%" has confidence 0.92.

**High confidence is not robustness.** Adversarial attacks are built to make the model *wrong*, and they often make it wrong *with high confidence*
(the proposal's abstract describes exactly this). A model can be very confident and still be fooled. What we want is a model that is **correct**
under attack and whose confidence is **honest**: when it says 80%, it should be right about 80% of the time.

We therefore report, for each model and each condition (clean, FGSM, PGD, transfer):
- mean confidence on **correctly** classified images,
- mean confidence on **incorrectly** classified images,
- the **high-confidence error rate**: the percentage of all images that are wrong *and* have confidence ≥ 0.9,
- the **Expected Calibration Error (ECE)**.

**ECE.** The images are grouped into `ECE_NUM_BINS` = 15 bins by confidence (0–0.067, 0.067–0.133, …). In each bin we compare the average confidence with
the real accuracy. ECE is the weighted average of the gaps:
$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{n_b}{N}\,\big|\,\mathrm{accuracy}_b - \mathrm{confidence}_b\,\big|$$
ECE = 0 means perfectly calibrated. A large ECE under attack with confidence > accuracy means the model is *over-confident* on adversarial images.
A **reliability diagram** draws the same comparison per bin: a perfectly calibrated model lies on the diagonal.

**15.1 ECE and confidence summary functions.**

In [ ]:
def expected_calibration_error(confidences, correct, num_bins=ECE_NUM_BINS):
    bin_edges = np.linspace(0.0, 1.0, num_bins + 1)
    ece = 0.0
    for i in range(num_bins):
        in_bin = (confidences > bin_edges[i]) & (confidences <= bin_edges[i + 1])
        if in_bin.sum() == 0:
            continue
        bin_accuracy = correct[in_bin].mean()
        bin_confidence = confidences[in_bin].mean()
        ece += in_bin.mean() * abs(bin_accuracy - bin_confidence)
    return 100.0 * ece   # in percent


def confidence_summary(result):
    correct = result["predictions"] == result["labels"]
    confidences = result["confidences"]
    return {
        "accuracy (%)": 100.0 * correct.mean(),
        "mean conf (all)": confidences.mean(),
        "mean conf (correct)": confidences[correct].mean() if correct.any() else np.nan,
        "mean conf (incorrect)": confidences[~correct].mean() if (~correct).any() else np.nan,
        "high-conf errors (%)": 100.0 * np.mean(~correct & (confidences >= 0.9)),
        "ECE (%)": expected_calibration_error(confidences, correct.astype(float)),
    }

**15.2 Confidence and calibration table** (all models, all conditions, $\epsilon = 8/255$).

In [ ]:
CONDITIONS = ["clean", "fgsm", "pgd", "fgsm_transfer", "pgd_transfer"]
CONDITION_LABELS = {"clean": "Clean", "fgsm": "FGSM white-box", "pgd": f"PGD-{EVAL_PGD_STEPS} white-box",
                    "fgsm_transfer": "FGSM transfer", "pgd_transfer": f"PGD-{EVAL_PGD_STEPS} transfer"}

confidence_rows = []
for run_key in evaluation_runs:
    for condition in CONDITIONS:
        if condition not in results[run_key]:
            continue
        row = {"model": run_label(run_key), "condition": CONDITION_LABELS[condition]}
        row.update(confidence_summary(results[run_key][condition]))
        confidence_rows.append(row)

confidence_table = pd.DataFrame(confidence_rows).set_index(["model", "condition"])
display(confidence_table.round(3))
confidence_table.to_csv(os.path.join(RESULTS_DIR, "confidence_calibration_table.csv"))

**15.3 Confidence on correct vs. incorrect predictions.**
For each model (first seed): mean confidence when it is right and when it is wrong, on clean and on PGD-attacked images.
A good model is confident when right and *unsure* when wrong.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
bar_width = 0.35
positions = np.arange(len(primary_runs))

for ax, condition in zip(axes, ["clean", "pgd"]):
    correct_means = [confidence_summary(results[k][condition])["mean conf (correct)"] for k in primary_runs]
    incorrect_means = [confidence_summary(results[k][condition])["mean conf (incorrect)"] for k in primary_runs]
    ax.bar(positions - bar_width / 2, correct_means, bar_width, label="correct predictions")
    ax.bar(positions + bar_width / 2, incorrect_means, bar_width, label="incorrect predictions")
    ax.set_xticks(positions)
    ax.set_xticklabels([run_label(k) for k in primary_runs], rotation=15)
    ax.set_title(f"Mean max-softmax confidence: {CONDITION_LABELS[condition]}")
    ax.set_ylim(0, 1)
axes[0].set_ylabel("confidence")
axes[0].legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confidence_correct_vs_incorrect.png"), dpi=150)
plt.show()

**15.4 Confidence histograms under PGD.**
How confident is each model on the PGD images it gets right (green) and wrong (red)? A tall red bar near 1.0 means the model is fooled *confidently*.

In [ ]:
fig, axes = plt.subplots(1, len(primary_runs), figsize=(4 * len(primary_runs), 3.5), sharey=True)
bins = np.linspace(0, 1, 21)

for ax, run_key in zip(axes, primary_runs):
    result = results[run_key]["pgd"]
    correct = result["predictions"] == result["labels"]
    ax.hist(result["confidences"][correct], bins=bins, alpha=0.7, color="green", label="correct")
    ax.hist(result["confidences"][~correct], bins=bins, alpha=0.7, color="red", label="incorrect")
    ax.set_title(run_label(run_key))
    ax.set_xlabel("confidence")
axes[0].set_ylabel("number of images")
axes[0].legend()
fig.suptitle(f"Confidence on PGD-{EVAL_PGD_STEPS} white-box images (eps = {round(EVAL_EPSILON * 255)}/255)")
plt.tight_layout()
plt.show()

**15.5 Reliability diagrams.**
Each bar is one confidence bin; its height is the real accuracy of the images in that bin. Bars below the dashed diagonal mean over-confidence.

In [ ]:
def plot_reliability_diagram(ax, result, title, num_bins=ECE_NUM_BINS):
    correct = (result["predictions"] == result["labels"]).astype(float)
    confidences = result["confidences"]
    bin_edges = np.linspace(0.0, 1.0, num_bins + 1)
    bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_accuracies = []
    for i in range(num_bins):
        in_bin = (confidences > bin_edges[i]) & (confidences <= bin_edges[i + 1])
        bin_accuracies.append(correct[in_bin].mean() if in_bin.sum() > 0 else 0.0)
    ax.bar(bin_centres, bin_accuracies, width=1.0 / num_bins, edgecolor="black", alpha=0.7)
    ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
    ece = expected_calibration_error(confidences, correct, num_bins)
    ax.set_title(f"{title}\nECE = {ece:.2f}%", fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)


fig, axes = plt.subplots(2, len(primary_runs), figsize=(4 * len(primary_runs), 8))
for column, run_key in enumerate(primary_runs):
    plot_reliability_diagram(axes[0][column], results[run_key]["clean"], f"{run_label(run_key)}: clean")
    plot_reliability_diagram(axes[1][column], results[run_key]["pgd"], f"{run_label(run_key)}: PGD-{EVAL_PGD_STEPS}")
    axes[1][column].set_xlabel("confidence")
axes[0][0].set_ylabel("accuracy")
axes[1][0].set_ylabel("accuracy")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "reliability_diagrams.png"), dpi=150)
plt.show()

---
# 16. Comparative Results and Visualisations

**16.1 Summary table for every trained model.**
One row per model and seed. Every robust-accuracy column names its attack, its source (white-box or transfer) and its budget, as the proposal requires.

In [ ]:
EPS_LABEL = f"eps={round(EVAL_EPSILON * 255)}/255"
summary_rows = []
for run_key in evaluation_runs:
    method, seed = run_key
    r = results[run_key]
    row = {
        "method": "Standard (undefended)" if method == "Standard" else method,
        "seed": seed,
        "clean acc (%)": accuracy_percent(r["clean"]),
        "clean loss": mean_loss(r["clean"]),
        f"FGSM white-box acc (%) {EPS_LABEL}": accuracy_percent(r["fgsm"]),
        f"FGSM white-box loss {EPS_LABEL}": mean_loss(r["fgsm"]),
        f"FGSM white-box ASR (%) {EPS_LABEL}": attack_success_rate(r["clean"], r["fgsm"]),
        f"PGD-{EVAL_PGD_STEPS} white-box acc (%) {EPS_LABEL}": accuracy_percent(r["pgd"]),
        f"PGD-{EVAL_PGD_STEPS} white-box loss {EPS_LABEL}": mean_loss(r["pgd"]),
        f"PGD-{EVAL_PGD_STEPS} white-box ASR (%) {EPS_LABEL}": attack_success_rate(r["clean"], r["pgd"]),
        f"FGSM transfer acc (%) {EPS_LABEL}": accuracy_percent(r["fgsm_transfer"]) if "fgsm_transfer" in r else np.nan,
        f"PGD-{EVAL_PGD_STEPS} transfer acc (%) {EPS_LABEL}": accuracy_percent(r["pgd_transfer"]) if "pgd_transfer" in r else np.nan,
        "ECE clean (%)": confidence_summary(r["clean"])["ECE (%)"],
        f"ECE PGD-{EVAL_PGD_STEPS} (%)": confidence_summary(r["pgd"])["ECE (%)"],
        "conf correct clean": confidence_summary(r["clean"])["mean conf (correct)"],
        f"conf correct PGD-{EVAL_PGD_STEPS}": confidence_summary(r["pgd"])["mean conf (correct)"],
        f"conf incorrect PGD-{EVAL_PGD_STEPS}": confidence_summary(r["pgd"])["mean conf (incorrect)"],
        "training time (h)": run_infos[run_key]["total_training_hours"],
    }
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(os.path.join(RESULTS_DIR, "summary_per_run.csv"), index=False)
display(summary_table.round(3))

**16.2 Comparison of the three defenses (mean and standard deviation over seeds).**
This is the main results table of the study. With only one seed, the standard deviation is empty (NaN); run with `SEEDS = [0, 1, 2]` to get it (Proposal 3.1 and 3.5).

In [ ]:
method_order = ["Standard (undefended)"] + METHOD_NAMES
numeric_columns = [column for column in summary_table.columns if column not in ["method", "seed"]]

comparison_mean = summary_table.groupby("method")[numeric_columns].mean().reindex(method_order)
comparison_std = summary_table.groupby("method")[numeric_columns].std().reindex(method_order)
seeds_per_method = summary_table.groupby("method")["seed"].count().reindex(method_order)

print("Number of seeds per method:")
display(seeds_per_method.to_frame("seeds"))
print("Mean over seeds:")
display(comparison_mean.T.round(3))
print("Standard deviation over seeds:")
display(comparison_std.T.round(3))

comparison_mean.to_csv(os.path.join(RESULTS_DIR, "comparison_mean.csv"))
comparison_std.to_csv(os.path.join(RESULTS_DIR, "comparison_std.csv"))

**16.3 Chart: clean vs. robust accuracy.**
For each method: accuracy on clean images, under white-box FGSM, under white-box PGD-20 and under transfer PGD-20 (mean over seeds; error bars = standard deviation).

In [ ]:
chart_columns = {
    "Clean": "clean acc (%)",
    "FGSM white-box": f"FGSM white-box acc (%) {EPS_LABEL}",
    f"PGD-{EVAL_PGD_STEPS} white-box": f"PGD-{EVAL_PGD_STEPS} white-box acc (%) {EPS_LABEL}",
    f"PGD-{EVAL_PGD_STEPS} transfer": f"PGD-{EVAL_PGD_STEPS} transfer acc (%) {EPS_LABEL}",
}
positions = np.arange(len(method_order))
bar_width = 0.2

fig, ax = plt.subplots(figsize=(11, 5))
for i, (label, column) in enumerate(chart_columns.items()):
    ax.bar(positions + (i - 1.5) * bar_width, comparison_mean[column], bar_width,
           yerr=comparison_std[column].fillna(0), capsize=3, label=label)
ax.set_xticks(positions)
ax.set_xticklabels(method_order)
ax.set_ylabel("accuracy (%)")
ax.set_title(f"Clean and robust accuracy by defense ({EPS_LABEL})")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "comparison_accuracy_chart.png"), dpi=150)
plt.show()

**16.4 Chart: robust accuracy as $\epsilon$ grows.**

In [ ]:
epsilon_labels = [round(e * 255) for e in EVAL_EPSILONS_SWEEP]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, attack in zip(axes, ["fgsm", "pgd"]):
    for run_key in primary_runs:
        accuracies = [results[run_key]["sweep"][f"{attack}_{e}"] for e in epsilon_labels]
        ax.plot(epsilon_labels, accuracies, marker="o", label=run_label(run_key))
    ax.set_xlabel("epsilon (x / 255)")
    ax.set_title("FGSM white-box" if attack == "fgsm" else f"PGD-{EVAL_PGD_STEPS} white-box")
    ax.grid(alpha=0.3)
axes[0].set_ylabel(f"accuracy (%) on {len(X_sweep)} test images")
axes[0].legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "accuracy_vs_epsilon.png"), dpi=150)
plt.show()

**16.5 Chart: the cost of robustness (Objective 3).**
Left: training time per method. Right: PGD-20 robust accuracy against training time: points further up and to the left give more robustness for less computation.

In [ ]:
pgd_column = f"PGD-{EVAL_PGD_STEPS} white-box acc (%) {EPS_LABEL}"
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(method_order, comparison_mean["training time (h)"], yerr=comparison_std["training time (h)"].fillna(0), capsize=3)
axes[0].set_ylabel("training time (hours)")
axes[0].set_title("Training time per model")
axes[0].tick_params(axis="x", rotation=15)

for method in method_order:
    axes[1].scatter(comparison_mean.loc[method, "training time (h)"], comparison_mean.loc[method, pgd_column], s=80)
    axes[1].annotate(method, (comparison_mean.loc[method, "training time (h)"], comparison_mean.loc[method, pgd_column]),
                     textcoords="offset points", xytext=(5, 5))
axes[1].set_xlabel("training time (hours)")
axes[1].set_ylabel(f"PGD-{EVAL_PGD_STEPS} white-box accuracy (%)")
axes[1].set_title("Robustness vs. training cost")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "robustness_vs_training_time.png"), dpi=150)
plt.show()

**16.6 Training and validation curves.**
One row per model (first seed). **Left:** losses. **Right:** accuracies.

- *Training* loss is each method's own objective (cross-entropy for Standard/SAT/Ensemble AT, cross-entropy + $\beta$·KL for TRADES), so training losses
  are **not** directly comparable between methods. Training accuracy is measured on each method's own (adversarial) training images.
- *Validation* clean and PGD-20 losses are the same cross-entropy for every model, so these **are** comparable.
- A validation PGD accuracy that rises and then falls while training accuracy keeps rising is a sign of *robust overfitting*.

In [ ]:
fig, axes = plt.subplots(len(primary_runs), 2, figsize=(13, 3.5 * len(primary_runs)))

for row, run_key in enumerate(primary_runs):
    history = training_histories[run_key]
    epochs = history["epoch"]

    axes[row][0].plot(epochs, history["train_loss"], label="train (method's own loss)")
    axes[row][0].plot(epochs, history["val_clean_loss"], label="validation clean (CE)")
    axes[row][0].plot(epochs, history["val_pgd_loss"], label=f"validation PGD-{EVAL_PGD_STEPS} (CE)")
    axes[row][0].set_title(f"{run_label(run_key)}: loss")
    axes[row][0].set_ylabel("loss")

    axes[row][1].plot(epochs, history["train_acc"], label="train")
    axes[row][1].plot(epochs, history["val_clean_acc"], label="validation clean")
    axes[row][1].plot(epochs, history["val_pgd_acc"], label=f"validation PGD-{EVAL_PGD_STEPS}")
    axes[row][1].set_title(f"{run_label(run_key)}: accuracy")
    axes[row][1].set_ylabel("accuracy (%)")

for ax in axes.flatten():
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_curves.png"), dpi=150)
plt.show()

**16.7 Classification reports.**
Precision, recall and F1-score per class for every model (first seed), on clean images and on white-box PGD-20 images.

In [ ]:
for run_key in primary_runs:
    for condition in ["clean", "pgd"]:
        result = results[run_key][condition]
        print(f"===== {run_label(run_key)} | {CONDITION_LABELS[condition]} =====")
        print(classification_report(result["labels"], result["predictions"], target_names=CLASSES, digits=3, zero_division=0))

**16.8 Confusion matrices.**
Row = true class, column = predicted class, normalised so each row sums to 1. The diagonal is the per-class accuracy.
Under attack, look for columns that collect many wrong predictions: classes that the attack "pushes" images towards.

In [ ]:
fig, axes = plt.subplots(2, len(primary_runs), figsize=(5 * len(primary_runs), 10))

for column, run_key in enumerate(primary_runs):
    for row, condition in enumerate(["clean", "pgd"]):
        result = results[run_key][condition]
        matrix = confusion_matrix(result["labels"], result["predictions"], labels=range(10), normalize="true")
        ConfusionMatrixDisplay(matrix, display_labels=CLASSES).plot(ax=axes[row][column], colorbar=False,
                                                                     values_format=".2f", xticks_rotation=90)
        axes[row][column].set_title(f"{run_label(run_key)}: {CONDITION_LABELS[condition]}", fontsize=10)
        for text in axes[row][column].texts:
            text.set_fontsize(6)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrices.png"), dpi=150)
plt.show()

**16.9 Per-class accuracy and the most common mistakes.**

In [ ]:
per_class_rows = []
for run_key in primary_runs:
    for condition in ["clean", "fgsm", "pgd"]:
        result = results[run_key][condition]
        row = {"model": run_label(run_key), "condition": CONDITION_LABELS[condition]}
        for class_index, class_name in enumerate(CLASSES):
            in_class = result["labels"] == class_index
            row[class_name] = 100.0 * np.mean(result["predictions"][in_class] == class_index)
        per_class_rows.append(row)

per_class_table = pd.DataFrame(per_class_rows).set_index(["model", "condition"])
display(per_class_table.round(1))
per_class_table.to_csv(os.path.join(RESULTS_DIR, "per_class_accuracy.csv"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
positions = np.arange(len(CLASSES))
bar_width = 0.8 / len(primary_runs)

for ax, condition in zip(axes, ["clean", "fgsm", "pgd"]):
    for i, run_key in enumerate(primary_runs):
        values = per_class_table.loc[(run_label(run_key), CONDITION_LABELS[condition])].values
        ax.bar(positions + i * bar_width, values, bar_width, label=run_label(run_key))
    ax.set_xticks(positions + 0.4 - bar_width / 2)
    ax.set_xticklabels(CLASSES, rotation=45)
    ax.set_title(f"Per-class accuracy: {CONDITION_LABELS[condition]}")
axes[0].set_ylabel("accuracy (%)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

Error analysis: the five most frequent (true class → predicted class) mistakes under white-box PGD-20 for each model.

In [ ]:
for run_key in primary_runs:
    result = results[run_key]["pgd"]
    matrix = confusion_matrix(result["labels"], result["predictions"], labels=range(10))
    np.fill_diagonal(matrix, 0)   # ignore correct predictions
    print(f"===== {run_label(run_key)} =====")
    flat_order = np.argsort(matrix, axis=None)[::-1][:5]
    for flat_index in flat_order:
        true_class, predicted_class = np.unravel_index(flat_index, matrix.shape)
        print(f"  {CLASSES[true_class]:>10s} -> {CLASSES[predicted_class]:<10s} {matrix[true_class, predicted_class]} images")

**16.10 Optional: TRADES $\beta$ sweep results.**
Clean and PGD-20 accuracy on `X_sweep` for each $\beta$ (only if the sweep was trained in Section 10.11). The main TRADES model ($\beta$ = `TRADES_BETA`) is included for reference.

In [ ]:
if RUN_TRADES_BETA_SWEEP:
    beta_rows = []
    beta_runs = [(beta, f"trades_beta{beta}_seed{primary_seed}") for beta in TRADES_BETA_SWEEP]
    beta_runs.append((TRADES_BETA, run_name_for("TRADES", primary_seed)))
    for beta, run_name in sorted(beta_runs):
        beta_model, _, _ = load_trained_model(run_name)
        torch.manual_seed(EVAL_SEED)
        clean_result = get_predictions(beta_model, X_sweep, y_sweep, attack="none")
        torch.manual_seed(EVAL_SEED)
        pgd_result = get_predictions(beta_model, X_sweep, y_sweep, attack="pgd")
        beta_rows.append({"beta": beta, "clean acc (%)": accuracy_percent(clean_result),
                          f"PGD-{EVAL_PGD_STEPS} acc (%) {EPS_LABEL}": accuracy_percent(pgd_result)})
    display(pd.DataFrame(beta_rows).set_index("beta").round(2))
else:
    print("Skipped (RUN_TRADES_BETA_SWEEP = False).")

---
# 17. Research Observations and Conclusions

**17.1 Automatic ranking from your results.**
This cell does not interpret anything. It only reads the comparison table from Section 16.2 and lists, for each metric, which of the three defenses
scored best. Use it as a starting point for your own written analysis.

In [ ]:
higher_is_better = {
    "clean acc (%)": True,
    f"FGSM white-box acc (%) {EPS_LABEL}": True,
    f"PGD-{EVAL_PGD_STEPS} white-box acc (%) {EPS_LABEL}": True,
    f"PGD-{EVAL_PGD_STEPS} white-box ASR (%) {EPS_LABEL}": False,
    f"PGD-{EVAL_PGD_STEPS} transfer acc (%) {EPS_LABEL}": True,
    f"ECE PGD-{EVAL_PGD_STEPS} (%)": False,
    "training time (h)": False,
}

defenses_only = comparison_mean.loc[METHOD_NAMES]
for metric, higher in higher_is_better.items():
    ranking = defenses_only[metric].sort_values(ascending=not higher)
    ranking_text = "  >  ".join(f"{method} ({value:.2f})" for method, value in ranking.items())
    print(f"{metric}\n    best to worst: {ranking_text}\n")

print(f"Seeds per defense: {seeds_per_method.loc[METHOD_NAMES].to_dict()}")
if len(SEEDS) == 1:
    print("Only one seed: differences of about 1 percentage point may be random variation.")

**17.2 Your observations** *(fill in after running the experiment; do not write conclusions before you have the numbers)*

Answer each question using the tables and charts above. Always state the attack, the setting (white-box / transfer) and $\epsilon$ next to any robust accuracy.

**Objective 1 (implementation and fair comparison)**
- Did all models train stably? (Section 16.6: look for diverging losses or robust overfitting.)
- Did every model pass the gradient-masking checks (Section 14.5)? If any `CHECK` appeared, what does it mean for that model's robust accuracy?

**Objective 2 (robustness)**
- Which defense has the highest white-box PGD-20 accuracy and the lowest ASR? Which is most robust to FGSM?
- How do the defenses rank under the **transfer** attack? Is the ranking different from the white-box ranking?
  (Proposal 3.4 expects Ensemble AT to be relatively strong against transfer attacks and weak against white-box iterative attacks. Do your results agree?)
- How does robustness fall as $\epsilon$ increases (Section 16.4)? Does any defense hold up better at larger budgets?

**Objective 3 (trade-offs)**
- How much clean accuracy does each defense give up compared with the undefended Standard model?
- How much training time does each defense cost (Section 16.5)? Which gives the best robustness per hour of training?

**Confidence and calibration**
- Under PGD, are wrong predictions made with high confidence (Sections 15.3–15.4)? Which model has the most high-confidence errors?
- Which model is best calibrated on clean images and under attack (ECE, reliability diagrams)? Remember: higher confidence is **not** the same as more robustness.

**Error analysis**
- Which classes are hardest under attack, and which class pairs are confused most (Sections 16.8–16.9)? Are they the same for every defense?

**Conclusion**
- *[Write 1–2 paragraphs: which defense you would recommend for which threat model (white-box vs transfer), and at what cost in clean accuracy and training time.]*

**17.3 Limitations and threats to validity** *(keep these in mind when writing the conclusion)*
- **One architecture and one dataset.** The results hold for `ResCNN` on CIFAR-10 and may not carry over to other models or data.
- **Attack strength.** FGSM and PGD-20 are standard but not the strongest attacks. Stronger benchmarks such as AutoAttack (Croce & Hein, 2020) and C&W are outside the scope of the proposal (Section 3.3).
- **Transfer source.** The transfer attacks come from one undefended model with the *same architecture*. Transfer from different architectures may be weaker or stronger.
- **Ensemble AT design choices.** The number of source models (2), using the same architecture for them, the single-step FGSM attack and including the current model in the rotation all follow Tramèr et al. as closely as this setting allows, but other choices are possible.
- **Seeds.** With a single seed, small differences between methods may not be meaningful; the proposal asks for several seeds.
- **Training time** depends on the GPU type Colab assigns. Compare times only between models trained on the same GPU (the GPU is recorded in each checkpoint).
- **Final-epoch models** are evaluated for every method (no early stopping), the same rule for all defenses.

---
# Appendix: Interactive Demos (from the original notebook)

These demos are kept from the original notebook and now use the new `ResCNN` models. They are for **demonstration only**; they are not part of the
experiment's measurements. They use the first-seed models and the same FGSM / PGD-20 functions as the evaluation.

**A.1 Prepare the demo models and the attack helpers.**
The demo compares four models: the Standard (undefended) model and the three defenses. The transfer image is crafted on the Standard model, as in the evaluation.

In [ ]:
from PIL import Image

STANDARD = "Standard"
models = {
    "Standard": trained_models[("Standard", BASELINE_SEED)],
    "SAT": trained_models[("SAT", primary_seed)],
    "Ensemble AT": trained_models[("Ensemble AT", primary_seed)],
    "TRADES": trained_models[("TRADES", primary_seed)],
}
PGD_STEPS = EVAL_PGD_STEPS


def craft(model, x, y, kind, eps):
    # Uses the same attack functions and settings as the evaluation.
    if kind == "FGSM":
        return fgsm_attack(model, x, y, eps)
    return pgd_attack(model, x, y, eps, step_size=eps * EVAL_PGD_STEP_FRACTION, num_steps=PGD_STEPS, random_start=True)


@torch.no_grad()
def predict(model, x):
    probs = F.softmax(model(x), dim=1)
    conf, pred = probs.max(1)
    return pred.item(), conf.item()


def prepare_photo(pil_image):
    """Turn any uploaded picture into the 32x32 tensor the models expect.
    The picture is centre-cropped to a square first (so it is not squashed), then shrunk to 32 x 32."""
    img = pil_image.convert("RGB")
    w, h = img.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    img = img.crop((left, top, left + side, top + side)).resize((32, 32), Image.LANCZOS)
    return transforms.ToTensor()(img).unsqueeze(0)          # shape (1, 3, 32, 32), values 0..1


def analyse(idx, eps, photo=None, true_label=None):
    """Attack one image against all four models.
    - Test-set image: pass idx (photo=None). The label is the real CIFAR-10 label.
    - Uploaded photo: pass photo (from prepare_photo). If you know its class, pass true_label (0-9).
      If you do not, the class the four models agree on most (average clean confidence) is used as the estimated label."""
    if photo is None:
        x = X_test[idx:idx+1].to(device)
        y = y_test[idx:idx+1].to(device)
    else:
        x = photo.to(device)
        if true_label is None:      # best estimate = average of the four models' clean answers
            with torch.no_grad():
                avg = sum(F.softmax(m(x), dim=1) for m in models.values()) / len(models)
            true_label = avg.argmax(1).item()
        y = torch.tensor([true_label], device=device)
    torch.manual_seed(0)
    transfer_img = craft(models[STANDARD], x, y, "PGD", eps)
    results_for_image = {}
    for name, m in models.items():
        images = {
            "Clean":  x,
            "FGSM":   craft(m, x, y, "FGSM", eps),
            "PGD-20": craft(m, x, y, "PGD", eps),
            "Transfer": None if name == STANDARD else transfer_img,
        }
        results_for_image[name] = {}
        for attack_name, img in images.items():
            if img is None:
                results_for_image[name][attack_name] = None
            else:
                pred, conf = predict(m, img)
                results_for_image[name][attack_name] = (img.cpu(), pred, conf)
    return int(y.item()), results_for_image

print("Attack tools ready.")

**A.2 Build the picture-maker.**
`report(idx, eps)` takes one test image (or, with `photo=...`, one uploaded photo) and draws one row per method and four columns: the clean image, then the FGSM, PGD-20 and transfer versions of it.
Every small picture is the exact image that method was shown, with its answer and confidence on top.
A **green** frame means the model answered correctly; a **red** frame means it was fooled. It also prints the same results as a table.

In [ ]:
ATTACKS = ["Clean", "FGSM", "PGD-20", "Transfer"]
HEADERS = {"Clean": "Clean image", "FGSM": "FGSM attack", "PGD-20": "PGD-20 attack",
           "Transfer": "Transfer attack\n(made on Standard)"}

def report(idx, eps=EVAL_EPSILON, photo=None, true_label=None):
    true, results_for_image = analyse(idx, eps, photo=photo, true_label=true_label)
    n = len(results_for_image)
    fig, axes = plt.subplots(n + 1, 4, figsize=(11, 2.8 * n + 0.8),
                             gridspec_kw={"height_ratios": [0.35] + [1] * n})
    for c, a in enumerate(ATTACKS):
        axes[0][c].axis("off")
        axes[0][c].text(0.5, 0.2, HEADERS[a], ha="center", va="center", fontsize=11, fontweight="bold")

    rows = []
    for r, (name, res) in enumerate(results_for_image.items(), start=1):
        row = {"Method": name}
        axes[r][0].set_ylabel(name, fontsize=11, fontweight="bold")
        for c, a in enumerate(ATTACKS):
            ax = axes[r][c]
            ax.set_xticks([]); ax.set_yticks([])
            item = res[a]
            if item is None:
                ax.text(0.5, 0.5, "not applicable\n(this model made\nthe attack)", ha="center", va="center", fontsize=9)
                for s in ax.spines.values():
                    s.set_visible(False)
                row[a] = "n/a"
                continue
            img, pred, conf = item
            ok = pred == true
            color = "green" if ok else "red"
            ax.imshow(img[0].permute(1, 2, 0).numpy(), interpolation="nearest")
            ax.set_title(f"{CLASSES[pred]} ({conf:.0%})", fontsize=10, color=color)
            for s in ax.spines.values():
                s.set_edgecolor(color); s.set_linewidth(3)
            row[a] = ("✓ " if ok else "✗ ") + f"{CLASSES[pred]} ({conf:.0%})"
        rows.append(row)

    if photo is None:
        source_text = f"Test image #{idx} | TRUE class: {CLASSES[true]}"
    elif true_label is not None:
        source_text = f"Your photo | TRUE class (you chose): {CLASSES[true]}"
    else:
        source_text = f"Your photo | ESTIMATED class (best guess from the 4 models): {CLASSES[true]}"
    fig.suptitle(f"{source_text} | attack strength ε = {eps:.4f} "
                 f"(about {round(eps * 255)}/255 per pixel)   [green = correct, red = fooled]",
                 fontsize=11, y=0.995)
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(rows).set_index("Method"))

print("Picture-maker ready.")

**A.3 Browse the test images.**
A small gallery: 6 example images for each of the 10 classes, each with its **number** (like `#127`). Pick one and remember its number for the next cell.

In [ ]:
idx_by_class = {c: [i for i, l in enumerate(test_set.targets) if l == c] for c in range(10)}

per_class = 6
fig, axes = plt.subplots(10, per_class, figsize=(per_class * 1.6, 10 * 1.7))
for c in range(10):
    for j in range(per_class):
        i = idx_by_class[c][j]
        ax = axes[c][j]
        ax.imshow(X_test[i].permute(1, 2, 0).numpy())
        ax.set_title(f"#{i}", fontsize=8)
        ax.axis("off")
    axes[c][0].text(-0.15, 0.5, CLASSES[c], transform=axes[c][0].transAxes,
                    ha="right", va="center", fontsize=10)
plt.tight_layout()
plt.show()

**A.4 Choose an image and test all four models on it.**
Use the form on the right of this cell (in Colab), then press play. You can choose the image in four ways:

- **random**: any test image
- **by class**: pick a class (for example `cat`); `example_number` picks which cat (0 is the first cat, 1 is the second, and so on)
- **by index**: type the exact `#number` you saw in the gallery
- **upload my own photo**: after you press play, a *Choose files* button appears under the cell. Pick a JPG or PNG from your computer.
  If you know what the photo shows, set `photo_true_class` (for example `cat`). If you leave it on `not sure`, the class the four models agree on most is used as the estimated class.

`epsilon` is the attack strength. Try 0.01 (weak), 0.031 (= 8/255, the evaluation budget) and 0.05 (strong).


In [ ]:
#@title Pick a CIFAR-10 test image, or upload your own photo
mode = "random" #@param ["random", "by class", "by index", "upload my own photo"]
class_name = "automobile" #@param ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
example_number = 34 #@param {type:"integer"}
test_index = 11 #@param {type:"integer"}
photo_true_class = "not sure" #@param ["not sure", "airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
epsilon = 0.031 #@param {type:"number"}

if mode == "upload my own photo":
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        photo_name = list(uploaded.keys())[0]
    else:
        photo_name = "my_photo.jpg"      # outside Colab: put the file next to the notebook
    my_photo = prepare_photo(Image.open(photo_name))
    chosen_label = None if photo_true_class == "not sure" else CLASSES.index(photo_true_class)
    report(None, epsilon, photo=my_photo, true_label=chosen_label)
else:
    if mode == "random":
        idx = random.randrange(len(test_set))
    elif mode == "by class":
        pool = idx_by_class[CLASSES.index(class_name)]
        idx = pool[example_number % len(pool)]
    else:
        idx = test_index
    report(idx, epsilon)


**A.5 Tips for uploading your own photo.**
Upload is now built into A.4 (mode `upload my own photo`) and into the web demo in A.6, so the photo goes through exactly the same attacks and picture-maker as a test image.

- The models only know the 10 CIFAR-10 classes, and the photo is centre-cropped to a square and shrunk to 32 × 32 pixels. A clear, close-up picture of one of those things works best.
- If a model gets the *clean* photo wrong, that is not an attack result. It means the photo is hard for the model, so set `photo_true_class` and judge the attack only where the clean answer is correct.


**A.6 Optional: Gradio web demo.**
The web app from the original notebook, now with a **photo upload box**. It uses `analyse` from A.1, so it shows the new `ResCNN` models and handles uploaded photos the same way as test images.

In [ ]:
!pip -q install gradio



import gradio as gr
import matplotlib.pyplot as plt
from html import escape
import traceback
import base64, io

APP_CSS = """
#encrypt-app {
    max-width: 1220px;
    margin: 0 auto;
}
.hero {
    padding: 30px 34px;
    border-radius: 22px;
    color: #f7f9fc;
    background: linear-gradient(120deg, #14233b, #203e63 58%, #496d91);
    margin-bottom: 20px;
    box-shadow: 0 12px 32px rgba(20, 35, 59, 0.18);
}
.hero h1 {
    margin: 0 0 8px 0;
    font-size: 30px;
    letter-spacing: -0.5px;
}
.hero p {
    margin: 0;
    color: #e2eaf4;
    font-size: 15px;
}
.hero-kicker {
    color: #c2d1e2;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    margin-bottom: 9px;
}
.hero-chips {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-top: 18px;
}
.hero-chip {
    color: #f3f6fb;
    background: rgba(255, 255, 255, 0.12);
    border: 1px solid rgba(255, 255, 255, 0.23);
    border-radius: 999px;
    padding: 6px 11px;
    font-size: 12px;
    font-weight: 650;
}
.control-panel,
.output-card {
    background: var(--block-background-fill) !important;
    border: 1px solid var(--block-border-color) !important;
    color: var(--body-text-color);
    border-radius: 18px !important;
    box-shadow: 0 8px 24px rgba(20, 35, 59, 0.06);
}
.control-panel {
    padding: 22px !important;
}
.output-card {
    padding: 16px !important;
    margin-bottom: 14px;
}
.status-box {
    min-height: 54px;
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 20px;
    text-align: left;
    color: var(--body-text-color);
    padding: 12px;
}
.status-loading {
    min-height: 150px;
    margin: 8px 0 14px;
    border: 1px solid var(--border-color-accent-subdued);
    border-radius: 16px;
    background: var(--background-fill-secondary);
}
.loader-ring {
    width: 54px;
    height: 54px;
    flex: 0 0 54px;
    border: 6px solid #d8e1ed;
    border-top-color: #254d78;
    border-right-color: #7895b5;
    border-radius: 50%;
    animation: spin 0.85s linear infinite;
    box-shadow: 0 0 22px rgba(37, 77, 120, 0.16);
}
.loading-title {
    color: var(--body-text-color);
    font-size: 18px;
    font-weight: 750;
}
.loading-copy {
    color: var(--body-text-color-subdued);
    margin-top: 4px;
}
.loading-dots {
    display: flex;
    gap: 5px;
    margin-top: 10px;
}
.loading-dots i {
    width: 7px;
    height: 7px;
    border-radius: 50%;
    background: #52769b;
    animation: pulse 1s ease-in-out infinite;
}
.loading-dots i:nth-child(2) { animation-delay: 0.15s; }
.loading-dots i:nth-child(3) { animation-delay: 0.3s; }
@keyframes spin {
    to { transform: rotate(360deg); }
}
@keyframes pulse {
    0%, 80%, 100% { opacity: 0.35; transform: scale(0.75); }
    40% { opacity: 1; transform: scale(1); }
}
.results-table-wrap {
    overflow-x: auto;
    border: 1px solid var(--block-border-color);
    border-radius: 13px;
}
.results-table {
    width: 100%;
    border-collapse: collapse;
    background: var(--block-background-fill);
    font-size: 14px;
}
.results-table th {
    background: var(--background-fill-secondary);
    color: var(--body-text-color);
    text-align: left;
    padding: 13px 14px;
    white-space: nowrap;
}
.results-table td {
    padding: 13px 14px;
    border-top: 1px solid var(--block-border-color);
    color: var(--body-text-color);
}
.results-table tr:hover td {
    background: var(--background-fill-secondary);
}
.method-name {
    color: var(--body-text-color) !important;
    font-weight: 700;
    white-space: nowrap;
}
.badge {
    display: inline-block;
    padding: 4px 9px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 700;
    white-space: nowrap;
}
.badge-correct {
    color: #ffffff;
    background: #15803d;
}
.badge-fooled {
    color: #ffffff;
    background: #b91c1c;
}
.badge-na {
    color: #536171;
    background: #e9edf2;
}
.legend,
.section-note {
    color: var(--body-text-color-subdued);
    font-size: 13px;
    line-height: 1.5;
}
.legend {
    margin: 10px 2px 2px;
}
.section-note {
    margin: -4px 0 12px;
}
"""

IDLE_STATUS = """
<div class="status-box">
    Choose an image and attack strength, then select <b>Run comparison</b>.
</div>
"""

def loading_status():
    return """
    <div class="status-box status-loading">
        <div class="loader-ring"></div>
        <div>
            <div class="loading-title">Preparing the comparison</div>
            <div class="loading-copy">
                Crafting the attacks and checking each defense…
            </div>
            <div class="loading-dots"><i></i><i></i><i></i></div>
        </div>
    </div>
    """

def photo_to_html(tensor_img, size=150):
    arr = (tensor_img[0].permute(1, 2, 0).numpy() * 255).astype("uint8")
    buf = io.BytesIO()
    Image.fromarray(arr).resize((size, size), Image.NEAREST).save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return (f'<img src="data:image/png;base64,{b64}" '
            f'style="width:{size}px;height:{size}px;image-rendering:pixelated;'
            f'border-radius:12px;border:1px solid #dce4ee"/>')


def build_summary(true_class, results, source_text, class_note):
    attacks = ["Clean", "FGSM", "PGD-20", "Transfer"]
    rows = ""
    for a in attacks:
        applicable = [it[a] for it in results.values() if it[a] is not None]
        ok = sum(1 for item in applicable if item[1] == true_class)
        rows += (f"<tr><td>{'Clean image' if a == 'Clean' else a + ' attack'}</td>"
                 f"<td><b>{ok} of {len(applicable)}</b> models answered correctly</td></tr>")

    held = [m for m, r in results.items() if r["PGD-20"][1] == true_class]
    verdict = ("Held up under PGD-20: <b>" + ", ".join(held) + "</b>") if held \
              else "<b>Every model was fooled by PGD-20.</b>"

    image_html = photo_to_html(results[STANDARD]["Clean"][0])
    return f"""
    <div style="display:flex;gap:22px;align-items:center;flex-wrap:wrap">
        <div>{image_html}<div style="font-size:12px;margin-top:4px">{escape(source_text)}</div></div>
        <div style="flex:1;min-width:260px">
            <div style="font-size:13px;opacity:.7">{escape(class_note)}</div>
            <div style="font-size:30px;font-weight:800;margin:2px 0 10px">{escape(CLASSES[true_class]).title()}</div>
            <table class="results-table"><tbody>{rows}</tbody></table>
            <div style="margin-top:10px">{verdict}</div>
        </div>
    </div>"""


def run_comparison(image_index, epsilon, uploaded_image=None, photo_class="Not sure"):
    if uploaded_image is not None:
        photo = prepare_photo(uploaded_image)
        label = None if photo_class == "Not sure" else CLASSES.index(photo_class)
        true_class, results = analyse(None, float(epsilon), photo=photo, true_label=label)
        source_text = "Your uploaded photo"
        class_word = "True class" if label is not None else "Estimated class"
        class_note = ("True class (you told the demo)" if label is not None else
                      "Estimated class: what the four models agree on for the clean photo")
    else:
        true_class, results = analyse(int(image_index), float(epsilon))
        source_text = f"Image #{int(image_index)}"
        class_word = "True class"
        class_note = "True class (official CIFAR-10 label)"

    attacks = ["Clean", "FGSM", "PGD-20", "Transfer"]
    titles = {
        "Clean": "Clean image",
        "FGSM": "FGSM",
        "PGD-20": "PGD-20",
        "Transfer": "Transfer attack",
    }
    methods = list(results.keys())

    fig, axes = plt.subplots(
        len(methods),
        len(attacks),
        figsize=(15, 3.2 * len(methods)),
        squeeze=False
    )

    table_rows = []

    for row_index, (method, attack_results) in enumerate(results.items()):
        cells = []

        for col_index, attack_name in enumerate(attacks):
            ax = axes[row_index, col_index]
            ax.set_xticks([])
            ax.set_yticks([])

            item = attack_results[attack_name]

            if item is None:
                ax.text(
                    0.5, 0.5, "Not applicable",
                    ha="center", va="center",
                    color="#64748b", fontsize=10
                )
                cells.append('<span class="badge badge-na">N/A</span>')
                image_title = titles[attack_name]
            else:
                image, prediction, confidence = item
                correct = prediction == true_class
                color = "#15803D" if correct else "#B91C1C"
                badge_class = "badge-correct" if correct else "badge-fooled"
                badge_text = "✓ Correct" if correct else "✕ Fooled"
                class_name = escape(CLASSES[prediction])

                ax.imshow(image[0].permute(1, 2, 0).numpy())

                cells.append(
                    f'<span class="badge {badge_class}">{badge_text}</span> '
                    f'{class_name} ({confidence:.0%})'
                )

                image_title = (
                    f"{titles[attack_name]}\n"
                    f"{class_name} · {confidence:.0%}"
                )

                for border in ax.spines.values():
                    border.set_visible(True)
                    border.set_color(color)
                    border.set_linewidth(2.5)

            if row_index == 0:
                ax.set_title(image_title, fontsize=9, pad=8, color=color if item is not None else "#536171")
            elif item is not None:
                ax.set_title(
                    f"{class_name} · {confidence:.0%}",
                    fontsize=9,
                    color=color,
                    pad=8
                )

            if col_index == 0:
                ax.set_ylabel(method, fontsize=10, fontweight="bold")

        table_rows.append(
            "<tr>"
            f'<td class="method-name">{escape(method)}</td>'
            + "".join(f"<td>{cell}</td>" for cell in cells)
            + "</tr>"
        )

    fig.suptitle(
        f"{source_text} · {class_word}: {escape(CLASSES[true_class])}"
        f" · Epsilon: {float(epsilon):.2f}",
        fontsize=14,
        fontweight="bold",
        y=0.995
    )
    fig.tight_layout(
        rect=[0.08, 0.02, 0.99, 0.95],
        pad=1.2,
        h_pad=1.5,
        w_pad=0.8
    )

    table_html = (
        '<div class="results-table-wrap">'
        '<table class="results-table">'
        '<thead><tr><th>Defense</th>'
        + "".join(f"<th>{titles[a]}</th>" for a in attacks)
        + "</tr></thead><tbody>"
        + "".join(table_rows)
        + "</tbody></table></div>"
        '<div class="legend">Green means the model kept the correct class. '
        'Red means the attack changed its prediction.</div>'
    )

    done_status = (
        '<div class="status-box">Comparison complete. '
        f'{class_word}: <b>{escape(CLASSES[true_class])}</b>'
        f' · {source_text}'
        f' · ε = {float(epsilon):.2f}</div>'
    )

    summary_html = build_summary(true_class, results, source_text, class_note)
    return fig, table_html, done_status, summary_html

def run_comparison_safe(image_index, epsilon, uploaded_image=None, photo_class="Not sure"):
    try:
        return run_comparison(image_index, epsilon, uploaded_image, photo_class)
    except Exception as error:
        traceback.print_exc()
        return (
            None,
            "",
            '<div class="status-box">'
            f'Comparison failed: {escape(str(error))}'
            '</div>',
            ""
        )

with gr.Blocks(
    title="Adversarial Attack Demo",
    elem_id="encrypt-app"
) as app:

    gr.HTML("""
<div class="hero">
    <div class="hero-kicker">CIFAR-10 · Project demonstration</div>
    <h1>Adversarial Attack Demo</h1>
    <p>
        See how small pixel changes affect image classification,
        and compare four training approaches.
    </p>
    <div class="hero-chips">
        <span class="hero-chip">4 defense methods</span>
        <span class="hero-chip">FGSM and PGD-20</span>
        <span class="hero-chip">White-box and transfer attacks</span>
        <span class="hero-chip">L∞ budget · default ε ≈ 8/255</span>
    </div>
</div>
""")

    with gr.Row():
        with gr.Column(
            scale=1,
            min_width=260,
            elem_classes=["control-panel"]
        ):
            gr.Markdown("## Experiment controls")

            image_control = gr.Slider(
                minimum=0,
                maximum=len(X_test) - 1,
                value=11,
                step=1,
                label="CIFAR-10 test image",
                info="Choose an image by its test-set number. Ignored while a photo is uploaded below."
            )

            upload_control = gr.Image(
                type="pil",
                label="Or upload your own photo",
                sources=["upload"],
                height=180
            )

            photo_class_control = gr.Dropdown(
                choices=["Not sure"] + list(CLASSES),
                value="Not sure",
                label="What does your photo show? (optional)",
                info="If unsure, the demo estimates the class from what the four models agree on."
            )

            epsilon_control = gr.Slider(
                minimum=0.01,
                maximum=0.05,
                value=0.03,
                step=0.01,
                label="Attack strength (epsilon)",
                info="Higher values allow a stronger pixel change."
            )

            run_button = gr.Button(
                "Run comparison",
                variant="primary",
                size="lg"
            )

            gr.Markdown(
                "The comparison evaluates the same image (a test image or your own photo) against "
                "Standard, SAT, Ensemble AT, and TRADES. Clear the uploaded photo (X) to go back to test images."
            )

        with gr.Column(scale=3, min_width=480):
            with gr.Group(elem_classes=["output-card"]):
                gr.Markdown("## Result summary")
                summary_output = gr.HTML()

            with gr.Group(elem_classes=["output-card"]):
                gr.Markdown("## Visual comparison")
                gr.Markdown(
                  '<div class="section-note">'
                  'Each row shows one defense. FGSM and PGD-20 are crafted against that row’s model. '
                  'The transfer example is crafted against the Standard model and tested against '
                  'the other defenses.'
                  '</div>'
                        )
                status_output = gr.HTML(value=IDLE_STATUS)
                plot_output = gr.Plot(label="Clean and attacked images")

            with gr.Group(elem_classes=["output-card"]):
                gr.Markdown("## Results by defense")
                table_output = gr.HTML()

    run_button.click(
        fn=loading_status,
        inputs=None,
        outputs=status_output,
        queue=False
    ).then(
        fn=run_comparison_safe,
        inputs=[image_control, epsilon_control, upload_control, photo_class_control],
        outputs=[plot_output, table_output, status_output, summary_output],
        show_progress="full"
    )

app.queue()
app.launch(
    share=True,
    inline=False,
    theme=gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="slate"
).set(
    body_background_fill="#f5f7fa",
    body_background_fill_dark="#0c1422",
    body_text_color="#26364a",
    body_text_color_dark="#e4ebf5",
    body_text_color_subdued="#607086",
    body_text_color_subdued_dark="#aab7c8",
    background_fill_primary="#ffffff",
    background_fill_primary_dark="#111c2d",
    background_fill_secondary="#edf2f8",
    background_fill_secondary_dark="#19263a",
    block_background_fill="#ffffff",
    block_background_fill_dark="#142137",
    block_border_color="#dce4ee",
    block_border_color_dark="#2b3b52",
    block_title_text_color="#263d59",
    block_title_text_color_dark="#dbe5f1"
),
    css=APP_CSS,
    debug=False
)